# 06 — Dự đoán AQI ngày mai: thời tiết có giúp hơn "ngày mai giống hôm nay" không?

Notebook này làm **câu phụ 2** (dự đoán AQI ngày mai, W10) và **câu phụ 3** (cảnh báo ngày AQI ≥ 201, W11) của đề tài. Nó dùng các quy ước của `03_eda` và kết quả của `05_weather_correlation`. Hàm dùng chung ở `src/model_utils.py` (mô hình, chỉ số, biểu đồ) và `src/forecast_utils.py` (dựng dữ liệu dự báo).

**Dữ liệu (đổi ở vòng sửa 2, F01).** Đặc trưng và nhãn được dựng **thẳng từ `data/raw/city_day.csv`** cùng bảng thời tiết theo ngày, theo protocol RA-DEC-02 (`reports/ra/round2/dev_label_feature_protocol.md`). Notebook **không** còn lấy đặc trưng hay nhãn từ bảng đã làm sạch hồi cứu (`air_quality_8cities*.csv`). Bảng đó được làm sạch bằng cách nhìn cả chuỗi (nội suy hai đầu, luật nghi lỗi top-5 và cửa sổ 15 ngày lấy tâm, số 0 treo), nên dùng nó làm đầu vào dự báo là để ngày sau lọt vào ngày trước. Bảng sạch hồi cứu chỉ còn được đọc ở ba chỗ, đều ghi rõ tại chỗ: bảng đối chiếu cohort (Phần A), Phần G (lặp lại cách làm của notebook 05) và phép so hai đường (Phần I).

**Trạng thái con số.** Các con số trong phần viết được đo trên snapshot đầu vào in ở ô Setup (sha256) và **đang chờ DA kiểm độc lập**; chưa phải kết quả đã được RA nghiệm thu.

**Cách chạy:** *Kernel -> Restart & Run All*, từ thư mục `notebooks/`. Không cần chạy `01` hay `04` trước: notebook đọc raw và bảng thời tiết baseline đã commit (`data/processed/weather_daily_8cities.csv`). Muốn dùng bảng thời tiết do `04_weather_merge` vừa tạo thì đặt biến môi trường `AIR_WEATHER_DAILY` trỏ tới file đó, hoặc chạy cả chuỗi bằng `python tools/run_chain.py --weather nb04` (xem `src/data_paths.py`). Hình lưu vào `reports/figures/` với tên `model_01_*` đến `model_05_*` (chạy ngay trong repo sẽ ghi đè 5 hình đã commit; để kiểm thử, chạy trong bản sao bằng `tools/run_chain.py`). Bảng tóm tắt lần chạy lưu vào `outputs/nb06/`. Môi trường đã kiểm: Python 3.11.2 với `requirements-lock.txt`.

## Câu hỏi nghiên cứu của đề tài

> **Thời tiết (gió, nhiệt độ, độ ẩm, mưa, lớp biên khí quyển) có liên quan thế nào tới các ngày ô nhiễm nặng (AQI ≥ 201) ở 8 thành phố Ấn Độ giai đoạn 2015–2020, và có giúp cảnh báo trước một ngày các ngày đó tốt hơn việc cho rằng "ngày mai giống hôm nay" không, để cơ quan môi trường và người dân có cảnh báo sức khỏe sớm hơn?**

Notebook 05 trả lời phần "liên quan thế nào". Notebook này trả lời phần "có giúp cảnh báo trước một ngày tốt hơn không", bằng cách so **cùng một mô hình** với ba mức thông tin tăng dần, trên dữ liệu mô hình chưa từng thấy.

## Ý chính: baseline "ngày mai = hôm nay" rất mạnh, nên mọi con số phải đọc kèm nó

AQI có tự tương quan cao: hôm nay nặng thì ngày mai thường nặng. Vì vậy câu hỏi đúng không phải "mô hình có chính xác không" mà là **"mô hình hơn baseline bao nhiêu điểm, và hơn có chắc không"**. Mọi so sánh dưới đây đều có baseline đi kèm và khoảng tin cậy 95% bootstrap theo khối 14 ngày lịch.

## Quy ước

| Quy ước | Giá trị |
|---|---|
| Một dòng | một (thành phố, **ngày hôm nay** *t*); mục tiêu = AQI của ngày *t*+1 |
| Dữ liệu | `forecast_utils.build_forecast_dataset()` (đọc `data/raw/city_day.csv` và `weather_daily_8cities.csv`), khoảng chung 14/06/2017 – 01/07/2020; giữ một mẫu khi raw có **cả** AQI hôm nay và AQI của ngày lịch kế tiếp. Nhãn là AQI **như được ghi trong raw**: không nội suy, không xoá theo luật nghi lỗi hồi cứu (RA-DEC-02 mục 2 và 3) |
| Chia | chia theo **ngày được dự đoán**. Train: đến 30/06/2019. **Test: 01/07/2019 – 29/02/2020** (trọn một mùa đông, 389 ngày nặng). 03–07/2020 (phong toả) tách riêng, không dùng để chọn gì |
| Chọn tham số | `TimeSeriesSplit` 4 lần **bên trong train**, cách nhau 2 ngày. Test chỉ mở một lần, ở Phần E |
| Mô hình chính | Ridge dự đoán **log AQI** ngày mai, báo lại theo điểm AQI. Mô hình cây (`HistGradientBoosting`) chỉ để so |
| Baseline | (1) trung bình theo thành phố; (2) trung vị cùng thành phố và cùng tháng (khí hậu); (3) **ngày mai = hôm nay** |
| Thời tiết | **chỉ đến hết hôm nay**. Thời tiết ngày mai chỉ xuất hiện ở "cận trên" (kịch bản dự báo hoàn hảo) |
| Chỉ số | hồi quy: MAE (chính), RMSE, R². Cảnh báo: recall, precision, F1, **không dùng accuracy** (W11) |
| Chống rò rỉ | đặc trưng của ngày *t* chỉ dùng raw và thời tiết đến hết ngày *t*; mọi imputer, scaler, encoder và mô hình nằm trong `Pipeline`, chỉ fit trên train; có phép thử rò rỉ **đi từ raw** ở Phần A |

## 0. Setup

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../src"))

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import scipy
import sklearn
import statsmodels
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.outliers_influence import variance_inflation_factor

import data_paths
import forecast_utils as fu
from model_utils import (CITIES, ACCENT, NEUTRAL, LEVELS, LEVEL_NAME, NEXT_FEATS, POLL_FEATS, CAL_FEATS,
                         TRAIN_END, TEST_START, TEST_END, LOCK_START, vn, set_style, build_dataset,
                         make_pipe, time_cv, cv_score, oof_predict, mae, rmse, r2, baseline_preds, design_matrix, coef_table,
                         alert_metrics, best_threshold, calendar_blocks, block_boot, boot_mae_diff, boot_alert_diff, boot_alert_ci,
                         finish_chart, save_fig)
from corr_utils import load_with_weather, core_cities, boot_corr, block_id

pd.set_option("display.max_colwidth", None, "display.width", 200)
set_style()
ENV = {"python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__,
       "scikit-learn": sklearn.__version__, "statsmodels": statsmodels.__version__, "matplotlib": matplotlib.__version__}
print("Môi trường:", " · ".join(f"{k} {v}" for k, v in ENV.items()))

# Đường dự báo đi thẳng từ RAW (vòng 2, F01): không đọc bảng đã làm sạch hồi cứu. Mỗi file đầu vào được in kèm sha256.
raw = fu.load_raw(verbose=True)                                # city_day.csv của 8 thành phố, đã kiểm khoá, chuỗi ngày và giá trị
wx = fu.load_weather_daily(verbose=True)                       # thời tiết theo ngày: baseline, hoặc file do AIR_WEATHER_DAILY chỉ định
ds = fu.build_forecast_dataset(raw, wx)                        # (thành phố, ngày hôm nay t) -> AQI raw ngày t+1, đã sắp theo thời gian
tr = ds[ds.split == "train"].reset_index(drop=True)
te = ds[ds.split == "test"].reset_index(drop=True)            # chỉ dùng ở Phần E trở đi
lk = ds[ds.split == "lockdown"].reset_index(drop=True)        # 03-07/2020, tách riêng (Phần H)

# Các con số mà phần viết của notebook nhắc tới được đo trên đúng các file đầu vào dưới đây (sha256). Chạy với file khác thì
# con số PHỤ THUỘC file đó được phép lệch: khi đó chúng chỉ được đối chiếu và ghi lại; con số chỉ phụ thuộc các file còn trùng
# snapshot vẫn phải khớp, và các bất biến (assert thường) phải đúng với mọi đầu vào. Bảng sạch hồi cứu KHÔNG đi vào đặc trưng,
# nhãn hay mô hình của đường dự báo: nó chỉ dùng cho Phần G và cho phép so hai đường.
SNAPSHOT = {"raw": "0d84b21c3e4878bbad8df362f2ab05f61ad959538dddf5918e714077ed3c1847",
            "thời tiết theo ngày": "45f3579a222c55c0156eb7571565bb7a818b89c64ad9b89eff97da02ef224c77",
            "bảng sạch hồi cứu (chỉ để đối chiếu)": "38653221fda73fedb9c1f6cac13953cec8f00b81c6628ab44b0bdcef03e7c0c7"}
INPUTS = {"raw": fu.RAW_FILE, "thời tiết theo ngày": fu.WEATHER_FILE, "bảng sạch hồi cứu (chỉ để đối chiếu)": data_paths.BASELINE_CLEAN}
chk = data_paths.SnapshotChecks(SNAPSHOT, {k: data_paths.sha256_file(v) for k, v in INPUTS.items()}, protocol=fu.PROTOCOL, defer=True)
chk.describe()                                                 # defer=True: chỗ lệch được gom lại, lỗi phát ra ở chk.finish() (ô mã cuối)

# Phần A — Dữ liệu, cách chia và kiểm tra rò rỉ (W10: chia trước, mọi thứ khác sau)

## Ba mức đặc trưng cho cùng một mô hình

Nền chung cho cả ba mức: **thành phố** (biến giả, Delhi là mức tham chiếu) và **mùa** (sin, cos của ngày được dự đoán). Mùa được đưa vào ngay từ mức 1 để phần "thêm thời tiết" không chỉ là thay thế cho mùa (mùa là confounder lớn nhất ở notebook 05). Phần I kiểm tra lại bằng cách bỏ hai biến mùa.

| Mức | Thêm vào | Ý nghĩa |
|---|---|---|
| 1 | log AQI hôm nay, hôm qua, hôm kia, trung bình 7 ngày trước | "Chỉ lịch sử AQI" |
| 2 | log(1 + nồng độ) hôm nay của PM2.5, NO2, CO, SO2, O3 | Biết ô nhiễm hôm nay do chất nào |
| 3 | nhiệt độ, độ ẩm, áp suất (lệch so với trung bình 30 ngày trước), đổi áp suất so với hôm qua, tốc độ gió, gió Đông, gió Bắc, lớp biên, log(1 + mưa) hôm nay | Thời tiết **đến hết hôm nay** |
| Cận trên | + cùng 9 biến thời tiết của **ngày mai** | Dự báo thời tiết hoàn hảo; không dùng được trong thực tế, chỉ cho biết trần |

## Ba quyết định thiết kế (nêu rõ vì là lựa chọn của nhóm, và đều được quyết trên train, không nhìn test)

1. **Log AQI.** AQI cao dao động mạnh hơn AQI thấp (dạng phễu ở hình 1); mô hình trên log AQI bỏ được dạng phễu (W10, sửa 1).
2. **Áp suất thành độ lệch so với trung bình 30 ngày trước của chính thành phố.** Áp suất tuyệt đối gần như là "tên thành phố" (Bengaluru cao 914 m nên thấp hơn hẳn): VIF lên tới 192 khi để nguyên. Độ lệch chỉ dùng quá khứ nên không rò rỉ.
3. **Chất ô nhiễm lấy thẳng từ raw, dựng lại cho từng ngày *t* chỉ bằng dữ liệu đến hết ngày *t*** (vòng sửa 2, RA-DEC-02 mục 3 và 4). Không có ô nội suy, vì nội suy tuyến tính dùng giá trị của các ngày *sau*. Không dùng luật nghi lỗi hồi cứu để xoá hay thay giá trị. Số 0 "treo" chỉ bị che khi **tính đến ngày *t*** đã thấy đủ 3 ngày 0 liên tiếp; ngày thứ nhất và thứ hai của một chuỗi 0 giữ nguyên số 0, vì lúc đó chưa thể biết chuỗi sẽ kéo dài. Quy tắc này chỉ áp cho PM2.5, NO2, CO, SO2, O3, không áp cho AQI và thời tiết. Ô thiếu (hoặc bị che) được điền bằng giá trị quan sát gần nhất của tối đa 2 ngày trước; số 0 đã bị che không được dùng làm nguồn điền. Phần I so đường này với đường hồi cứu cũ.

Giá trị còn thiếu (AQI các ngày trước, chất ô nhiễm) được điền bằng trung vị **của train** bên trong `Pipeline`, kèm cột báo thiếu.

In [ ]:
rows = []
for c in CITIES:
    r = {"Thành phố": c}
    for s_, lab in [("train", "train"), ("test", "test"), ("lockdown", "03-07/2020")]:
        x = ds[(ds.City == c) & (ds.split == s_)]
        r[f"{lab}: n"], r[f"{lab}: ngày nặng"] = len(x), int(x.y_poor.sum())
    rows.append(r)
split_tab = pd.DataFrame(rows).set_index("Thành phố")
split_tab.loc["Cộng"] = split_tab.sum()
display(split_tab)
print(f"Tỉ lệ ngày AQI >= 201: train {tr.y_poor.mean()*100:.1f}%, test {te.y_poor.mean()*100:.1f}%, 03-07/2020 {lk.y_poor.mean()*100:.1f}%")

# Bảng giữ mẫu: mỗi cặp (thành phố, ngày t) của khoảng chung rơi vào đúng một trong ba ô (thiếu AQI hôm nay / thiếu nhãn / giữ)
ret = fu.retention(ds)
print("\nBảng giữ mẫu (mẫu được giữ khi raw có AQI ở CẢ ngày t và ngày lịch t+1):")
display(ret)
assert (ret["cặp (thành phố, ngày t) trong khoảng"] == ret[["thiếu AQI(t)", "có AQI(t), thiếu AQI(t+1)", "giữ (có cả hai)"]].sum(axis=1)).all()
assert ret.loc["Cộng", "giữ (có cả hai)"] == len(ds) and (split_tab.loc["Cộng", ["train: n", "test: n", "03-07/2020: n"]] == [len(tr), len(te), len(lk)]).all()

# Đếm lại ĐỘC LẬP từ file raw bằng pandas thuần (không qua forecast_utils): số cặp có AQI ở cả ngày t và ngày lịch t+1
r0 = pd.read_csv(fu.RAW_FILE, usecols=["City", "Date", "AQI"], parse_dates=["Date"])
r0 = r0[r0.City.isin(CITIES) & r0.AQI.notna()]
pairs = r0.merge(r0.assign(Date=r0.Date - pd.Timedelta(days=1)), on=["City", "Date"])
pairs = pairs[(pairs.Date >= fu.COMMON_START) & (pairs.Date + pd.Timedelta(days=1) <= fu.COMMON_END)]
assert set(zip(pairs.City, pairs.Date)) == set(zip(ds.City.astype(str), ds.Date)) and len(pairs) == len(ds), "cohort khác với số đếm lại từ raw"

# So với cohort của đường hồi cứu cũ (bảng đã làm sạch): khoá nào thêm, khoá nào mất, và vì sao. Không giả định trước số lượng.
ds_old = build_dataset(verbose=False)
cdiff = fu.cohort_diff(ds, ds_old, raw)
n_new, n_lost = int((cdiff.cohort == "chỉ có ở đường dự báo (raw)").sum()), int((cdiff.cohort == "chỉ có ở đường hồi cứu").sum())
print(f"\nSo với đường hồi cứu cũ ({len(ds_old):,} mẫu): đường dự báo từ raw có {len(ds):,} mẫu; {n_new} mẫu chỉ có ở đường mới, {n_lost} mẫu chỉ có ở đường cũ.")
display(cdiff)

# Con số của snapshot hiện có (chỉ phụ thuộc file raw; phần viết bên dưới nhắc tới chúng). Đúng snapshot thì phải khớp.
_ = chk.expect((len(tr), len(te), len(lk)) == (5746, 1947, 975), f"số mẫu train/test/03-07/2020 = {(len(tr), len(te), len(lk))}, snapshot (5746, 1947, 975)", needs=["raw"])
_ = chk.expect((int(tr.y_poor.sum()), int(te.y_poor.sum()), int(lk.y_poor.sum())) == (1436, 389, 18),
               f"ngày nặng train/test/03-07/2020 = {(int(tr.y_poor.sum()), int(te.y_poor.sum()), int(lk.y_poor.sum()))}, snapshot (1436, 389, 18)", needs=["raw"])
_ = chk.expect((n_new, n_lost) == (5, 0) and (cdiff.split == "train").all(),
               f"mẫu chỉ có ở đường mới / chỉ có ở đường cũ = {(n_new, n_lost)}, snapshot (5, 0), đều thuộc train",
               needs=["raw", "bảng sạch hồi cứu (chỉ để đối chiếu)"])

In [ ]:
# 1. thứ tự thời gian: ngày được dự đoán của train < test < phong toả, không giao nhau
assert tr.Date_target.max() < te.Date_target.min() and te.Date_target.max() < lk.Date_target.min()
assert (ds.Date_target == ds.Date + pd.Timedelta(days=1)).all()
# 2. các mức 1-3 không chứa thời tiết ngày mai; bảng đặc trưng as-of không có cột nào của ngày t+1 (nhãn, tập, thời tiết ngày mai)
assert not set(NEXT_FEATS) & set(LEVELS["L3"])
feat = fu.make_forecast_features(raw, wx)
assert not {"y", "y_log", "y_poor", "split", *NEXT_FEATS} & set(feat.columns)
# 3. nhãn = AQI raw của NGÀY LỊCH t+1, lấy thẳng từ file raw: không dịch theo dòng, không nội suy, không điền
lab = r0.set_index(["City", "Date"]).AQI
assert (ds.y.to_numpy() == lab.reindex(pd.MultiIndex.from_arrays([ds.City.astype(str), ds.Date_target])).to_numpy()).all()
assert (ds.AQI.to_numpy() == lab.reindex(pd.MultiIndex.from_arrays([ds.City.astype(str), ds.Date])).to_numpy()).all()
# 4. phép thử rò rỉ ĐI TỪ RAW. Với mỗi ngày cắt t: (a) dựng lại đặc trưng từ raw + thời tiết chỉ gồm các ngày <= t;
#    (b) dựng lại từ dữ liệu mà MỌI giá trị sau t đã bị làm nhiễu (kể cả tạo chuỗi số 0 ngay sau t). So đặc trưng của ngày t
#    (AQI, toàn bộ cột mức 1-3, nguồn forward-fill, cờ số 0 treo; 8 thành phố) với bản đầy đủ, không dung sai.
CUTS = ["2017-06-14", "2017-07-08", "2017-07-09", "2017-07-30", "2017-08-12", "2017-08-29", "2017-08-30", "2017-11-09", "2017-11-13",
        "2018-02-06", "2018-06-15", "2018-09-15",                      # 12 ngày mà bảng sạch hồi cứu đã đổi khi cắt raw (DA, V05)
        "2017-07-14", "2017-07-16", "2018-05-01",                      # ba ngày có đặc trưng khác trong walk-forward của DA (V07)
        "2018-12-01", "2019-11-15", "2020-01-10",                      # ba ngày của phép thử cũ
        "2019-06-29", "2019-06-30", "2020-02-28", "2020-02-29", "2020-06-30"]   # hai bên ranh train/test, test/03-07/2020, và ngày t cuối
leak = fu.asof_check(CUTS, raw, wx)
display(leak)
assert (leak["số thành phố"] == 8).all() and len(leak) == len(CUTS)
assert (leak[["ô khác (cắt raw ở t)", "ô khác (nhiễu dữ liệu sau t)"]] == 0).all().all(), "đặc trưng của ngày t phụ thuộc vào dữ liệu sau t"
# 5. thời tiết ngày mai (chỉ dùng ở "cận trên") và nhãn là NaN ở hàng cuối của bản cắt: chúng không tồn tại vào ngày t
cut = pd.Timestamp("2019-11-15")
last = fu.attach_target(fu.make_forecast_features(raw, wx, as_of=cut), raw[raw.Date <= cut])
last = last[last.Date == cut]
assert len(last) == 8 and last[NEXT_FEATS].isna().all().all() and last.y.isna().all()
print(f"Đạt cả 5 kiểm tra: tại {len(CUTS)} ngày thử, đặc trưng của ngày t không đổi một ô nào khi cắt raw ở t hoặc làm nhiễu dữ liệu sau t.")

### Đọc bảng và kiểm tra

- Tập test có đúng **389 ngày nặng** trên 1.947 cặp (thành phố, ngày), trọn một mùa đông. Phong toả (03–07/2020) chỉ có **18** ngày nặng, nên chỉ dùng để xem sai số, không tính recall và precision.
- **Tỉ lệ ngày nặng giảm dần theo thời gian**: 25,0% (train) → 20,0% (test) → 1,8% (phong toả). Đây là xu hướng đã thấy ở `03_eda`. Hệ quả: ngưỡng chọn trên train có thể hơi "dễ" hơn so với test.
- Ba thành phố miền Nam gần như không có ngày nặng (Bengaluru, Hyderabad: 0; Chennai: 10 trong test). **Không đánh giá được cảnh báo ở đó**; chỉ có sai số AQI.
- **Bảng giữ mẫu.** Trong 8.904 cặp (thành phố, ngày *t*) của khoảng chung, 174 cặp thiếu AQI hôm nay và 62 cặp có AQI hôm nay nhưng thiếu AQI của ngày lịch kế tiếp; còn lại 8.668 mẫu. Số này trùng với phép đếm lại độc lập từ file raw ở ô trên. Chỉ đánh giá được những ngày có nhãn: đây là **giới hạn về độ phủ** của phép đánh giá.
- **So với đường hồi cứu cũ (8.663 mẫu): thêm 5 mẫu, đều ở train, không mất mẫu nào; test và 03–07/2020 giữ nguyên khoá.** Cả 5 là các cặp mà bảng sạch hồi cứu đã xoá AQI hôm nay hoặc AQI ngày mai bằng luật nghi lỗi (bảng trên ghi khoá, AQI raw và lý do). Đường dự báo giữ chúng vì luật đó nhìn cả chuỗi. Giữ một điểm nghi lỗi chưa được xác minh **không** có nghĩa là khẳng định điểm đó đúng: nhãn ở đây là "AQI được ghi trong raw của snapshot hiện có", không phải số đo đã được kiểm định tại trạm.
- **Phép thử rò rỉ đi từ raw** (ô trên): với mỗi ngày cắt *t*, dựng lại đặc trưng từ raw và thời tiết chỉ gồm các ngày ≤ *t*, rồi dựng lại lần nữa sau khi làm nhiễu mọi giá trị sau *t*. Đặc trưng của ngày *t* (AQI, mọi cột mức 1–3, nguồn điền, cờ số 0 treo; 8 thành phố) **trùng từng ô** với bản đầy đủ ở cả 23 ngày thử, gồm các ngày mà bảng sạch hồi cứu từng đổi khi cắt raw (DA V05, V07) và hai bên các ranh chia tập. Nhãn và thời tiết ngày mai là NaN ở hàng cuối của bản cắt. Phép thử phủ mọi ngày của khoảng chung nằm ở `tests/test_forecast_leakage.py`.

# Phần B — Baseline và chọn tham số, chỉ trên validation (chưa đụng test)

`TimeSeriesSplit` cho 4 lần chia, mỗi lần fit trên quá khứ và đo trên khoảng liền sau. Bảng đầu cho thấy validation phủ cả mùa đông 2017–18, mùa mưa 2018 và mùa đông 2018–19. Bảng sau là MAE trung bình qua 4 lần chia (cột "± giữa các lần chia" là độ lệch chuẩn giữa các lần). Tham số `alpha` của Ridge và cấu hình cây được chọn bằng MAE này.

Đừng so MAE ở đây với MAE ở Phần E: validation (2017–2019) và test (07/2019–02/2020) là hai giai đoạn khác nhau.

In [ ]:
folds = list(time_cv(len(tr)).split(tr))
fold_tab = pd.DataFrame([{"lần": k + 1, "fit trên ngày được dự đoán đến": f"{tr.iloc[a].Date_target.max():%d/%m/%Y}",
                          "validation": f"{tr.iloc[b].Date_target.min():%d/%m/%Y} - {tr.iloc[b].Date_target.max():%d/%m/%Y}",
                          "n fit": len(a), "n validation": len(b), "ngày nặng (validation)": int(tr.iloc[b].y_poor.sum())}
                         for k, (a, b) in enumerate(folds)]).set_index("lần")
display(fold_tab)

cv_base = {}
for a, b in folds:
    for name, p in baseline_preds(tr.iloc[a], tr.iloc[b]).items():
        cv_base.setdefault(name, []).append(mae(tr.iloc[b].y, p))

In [ ]:
ALPHAS = [0.1, 1, 10, 30, 100, 300]
cv_r = {lv: {a: cv_score(tr, f, lambda a=a, f=f: make_pipe(f, "ridge", alpha=a)) for a in ALPHAS} for lv, f in LEVELS.items()}
best_alpha = {lv: min(v, key=lambda a: v[a][0]) for lv, v in cv_r.items()}

HGB_GRID = [dict(max_depth=d, max_iter=n, learning_rate=0.05, min_samples_leaf=40) for d in (2, 3) for n in (100, 200)]
cv_h = {lv: [cv_score(tr, LEVELS[lv], lambda kw=kw, f=LEVELS[lv]: make_pipe(f, "hgb", **kw)) for kw in HGB_GRID] for lv in ["L1", "L2", "L3"]}
best_hgb = {lv: HGB_GRID[int(np.argmin([m for m, _ in v]))] for lv, v in cv_h.items()}

rows = [{"mô hình": k, "MAE validation": np.mean(v), "± giữa các lần chia": np.std(v), "tham số chọn": "—"} for k, v in cv_base.items()]
rows += [{"mô hình": f"Ridge {lv}: {LEVEL_NAME[lv]}", "MAE validation": cv_r[lv][best_alpha[lv]][0], "± giữa các lần chia": cv_r[lv][best_alpha[lv]][1],
          "tham số chọn": f"alpha = {best_alpha[lv]}"} for lv in LEVELS]
rows += [{"mô hình": f"Cây tăng cường (HGB) {lv}", "MAE validation": min(m for m, _ in cv_h[lv]), "± giữa các lần chia": cv_h[lv][int(np.argmin([m for m, _ in cv_h[lv]]))][1],
          "tham số chọn": f"depth {best_hgb[lv]['max_depth']}, {best_hgb[lv]['max_iter']} cây"} for lv in ["L1", "L2", "L3"]]
cv_tab = pd.DataFrame(rows).set_index("mô hình")
display(cv_tab.round(2))

### Đọc bảng

- **"Ngày mai = hôm nay" là baseline rất mạnh**: MAE 26,0 điểm, so với 60,5 của "luôn đoán trung bình thành phố" và 56,1 của khí hậu cùng tháng. Hai baseline kia gần như vô dụng cho dự báo một ngày, vì AQI đổi theo ngày nhiều hơn theo mùa.
- **Mô hình chỉ có lịch sử AQI (mức 1) không hơn baseline**: 25,5 so với 26,0, chênh nhỏ hơn độ lệch giữa các lần chia (1,4). Thêm chất ô nhiễm (mức 2) xuống 23,7, thêm thời tiết (mức 3) xuống 22,3.
- Cây tăng cường cho kết quả **tương đương Ridge ở mức 3** (22,4 và 22,3). (Ở mức 2 cây hơn Ridge khoảng 0,8 điểm; ở mức 1 thì kém.) Với mức 3, validation không có lý do chọn cây, nên mô hình chính vẫn là Ridge, đúng như đã định từ trước (W10: "bắt đầu bằng mô hình tuyến tính, giữ nó nếu gần bằng").
- `alpha` chọn được nhỏ đến vừa (0,1 đến 30) và MAE gần như phẳng trong khoảng đó: kết quả không nhạy với lựa chọn này.

# Phần C — Chẩn đoán mô hình trước khi tin nó (W10: LINE, VIF, Cook)

Dùng `statsmodels` OLS trên **train**, mức 3, mục tiêu log AQI, chỉ các dòng đủ đặc trưng (để có hệ số diễn giải được; Ridge dùng để dự đoán thì điền trung vị trong `Pipeline`). Hình 1 so mô hình trên AQI với mô hình trên log AQI.

In [ ]:
f3 = LEVELS["L3"]
cc = tr.dropna(subset=f3).reset_index(drop=True)                       # chỉ để diễn giải: OLS cần dòng đủ đặc trưng
X = design_matrix(cc, f3)
grp = ((cc.Date_target - cc.Date_target.min()).dt.days // 14).to_numpy()   # khối 14 ngày lịch, chung cho mọi thành phố
m_raw = sm.OLS(cc.y, X).fit()                                          # mục tiêu là AQI
m_log = sm.OLS(cc.y_log, X).fit()                                      # mục tiêu là log AQI
m_rob = sm.OLS(cc.y_log, X).fit(cov_type="cluster", cov_kwds={"groups": grp})   # cùng hệ số, sai số chuẩn bền theo khối
print(f"Dòng đủ đặc trưng: {len(cc)}/{len(tr)} ({len(cc)/len(tr)*100:.0f}%) · R² hiệu chỉnh trong mẫu (log AQI): {m_log.rsquared_adj:.3f} (KHÔNG phải thước đo chất lượng, xem Phần E)")

bp_raw, bp_log = het_breuschpagan(m_raw.resid, m_raw.model.exog), het_breuschpagan(m_log.resid, m_log.model.exog)
funnel = lambda m: pd.Series(np.abs(m.resid)).corr(pd.Series(m.fittedvalues), method="spearman")
acf1 = (cc.assign(r=m_log.resid.values).sort_values("Date").groupby("City", observed=True).r.apply(lambda s: s.autocorr(1)))
diag = pd.DataFrame({"mục tiêu = AQI": {"Spearman(|phần dư|, giá trị khớp)": funnel(m_raw), "Breusch-Pagan LM": bp_raw[0], "Durbin-Watson": durbin_watson(m_raw.resid)},
                     "mục tiêu = log AQI": {"Spearman(|phần dư|, giá trị khớp)": funnel(m_log), "Breusch-Pagan LM": bp_log[0], "Durbin-Watson": durbin_watson(m_log.resid)}})
display(diag.round(2))
print("Tự tương quan bậc 1 của phần dư (log AQI), theo thành phố:"); display(acf1.round(2).to_frame("ACF lag 1").T)

vif = pd.Series([variance_inflation_factor(X.values, i) for i in range(X.shape[1])], index=X.columns).drop("const").sort_values(ascending=False)
print("VIF (W10: > 5 đáng xem, > 10 nghiêm trọng):"); display(vif.round(1).head(8).to_frame("VIF").T)
cook = m_log.get_influence().cooks_distance[0]
print(f"Cook's distance: {int((cook > 4/len(cook)).sum())} dòng vượt 4/n ({(cook > 4/len(cook)).mean()*100:.1f}%), lớn nhất {cook.max():.3f}")

## Biểu đồ 1: log AQI bỏ được dạng phễu, nhưng phương sai vẫn chưa đều

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11.5, 8))
ax = axes[0, 0]
ax.scatter(m_raw.fittedvalues, m_raw.resid, s=5, alpha=0.35, color=NEUTRAL)
ax.axhline(0, color="#555555", lw=1); ax.set_xlabel("AQI khớp (điểm)"); ax.set_ylabel("Phần dư (điểm AQI)")
ax.text(0.02, 0.95, "Mục tiêu = AQI", transform=ax.transAxes, va="top", fontsize=9, fontweight="bold")
ax = axes[0, 1]
ax.scatter(m_log.fittedvalues, m_log.resid, s=5, alpha=0.35, color=ACCENT)
ax.axhline(0, color="#555555", lw=1); ax.set_xlabel("log AQI khớp"); ax.set_ylabel("Phần dư (log AQI)")
ax.text(0.02, 0.95, "Mục tiêu = log AQI (mô hình dùng)", transform=ax.transAxes, va="top", fontsize=9, fontweight="bold")
ax = axes[1, 0]
(osm, osr), (slope, icpt, _) = stats.probplot(m_log.resid, dist="norm")
ax.scatter(osm, osr, s=5, alpha=0.4, color=ACCENT); ax.plot(osm, icpt + slope * osm, color="#555555", lw=1)
ax.set_xlabel("Phân vị lý thuyết (chuẩn)"); ax.set_ylabel("Phân vị phần dư (log AQI)")
ax = axes[1, 1]
wk = cc.assign(r=m_log.resid.values, w=cc.Date_target.dt.to_period("W").dt.start_time).groupby("w").r.mean()
ax.plot(wk.index, wk.values, color=ACCENT, lw=1.2); ax.axhline(0, color="#555555", lw=1)
ax.set_xlabel("Tuần (ngày được dự đoán)"); ax.set_ylabel("Trung bình tuần của phần dư\n(8 thành phố, log AQI)")
assert m_raw.fittedvalues.min() < 0, "Mô hình trên AQI không còn khớp ra AQI âm: sửa phần đọc biểu đồ 1"
assert funnel(m_raw) > 0.15 > abs(funnel(m_log)), "Dạng phễu đã đổi: sửa tiêu đề và phần đọc kết quả"
finish_chart(fig, axes[0, 0], f"Mô hình log AQI hết dạng phễu: tương quan giữa độ lớn phần dư và giá trị khớp giảm từ {vn(funnel(m_raw), 2)} xuống {vn(funnel(m_log), 2).replace('-', '−')}; "
                              f"phương sai vẫn chưa đều (Breusch-Pagan) nên dùng sai số chuẩn bền", tr,
             note="Chỉ dùng tập train (OLS, mức 3, dòng đủ đặc trưng). Mỗi chấm là một cặp (thành phố, ngày).")
save_fig(fig, "01_residual_diagnostics")


### Đọc biểu đồ 1 và bảng chẩn đoán

- **Linearity.** Mô hình trên AQI (trên trái) cho phần dư thành một dải cong và còn khớp ra **AQI âm** (đến khoảng −150), vô lý về mặt vật lý. Mô hình trên log AQI (trên phải) là đám mây không có hình dạng quanh 0: đạt.
- **Equal variance.** Trên AQI, độ lớn phần dư tăng theo giá trị khớp (Spearman 0,23): dạng phễu. Trên log AQI còn −0,03: phễu biến mất. **Nhưng Breusch-Pagan vẫn bác bỏ phương sai đều** (LM 316, từ 420) vì phương sai còn khác nhau theo thành phố và mùa. Hệ quả (W10): hệ số không lệch nhưng sai số chuẩn thì sai, nên mọi khoảng tin cậy ở Phần D dùng **sai số chuẩn bền theo khối 14 ngày**, không dùng p-value mặc định.
- **Normality.** QQ-plot (dưới trái) khá thẳng ở giữa, **đuôi phải nặng**: có những ngày AQI tăng vọt mà mô hình không báo trước (phần dư log lên tới +1,2, tức gấp ~3 lần dự đoán). Đây là tiêu chí ít quan trọng nhất của LINE, nhưng là thông tin thật: mô hình thường **hụt ở các đợt tăng đột ngột**.
- **Independence.** Durbin-Watson 1,84 và tự tương quan bậc 1 của phần dư chỉ 0,01 đến 0,19 ở bảy thành phố: lịch sử AQI đã hấp thụ gần hết tự tương quan. **Ngoại lệ Hyderabad (0,34)**: ở đó mô hình còn bỏ sót quán tính. Hình dưới phải không có xu hướng hay sóng dài.
- **VIF.** Bốn biến AQI trễ có VIF 9,8 đến 14,5 (chúng vốn là cùng một đại lượng đo ở các ngày kề nhau), nhiệt độ 8,7 và PM2.5 8,3. Hệ quả: **không diễn giải từng hệ số của AQI trễ riêng lẻ** (hệ số của AQI hôm qua âm chỉ vì nó đang "sửa" hệ số của hôm nay). Ridge xử lý được điều này khi dự đoán.
- **Cook's distance.** Lớn nhất 0,077, không có dòng nào chi phối mô hình (345 dòng vượt 4/n là bình thường khi n lớn và phần dư có đuôi nặng).
- **R² hiệu chỉnh trong mẫu 0,89 không phải thước đo chất lượng** (W10): phần lớn là do AQI hôm nay đã "giải thích" AQI ngày mai. Chất lượng chỉ đọc ở Phần E, trên test và so với baseline.

# Phần D — Diễn giải hệ số: mỗi hệ số một câu (W10)

Mỗi câu đủ bốn nghĩa vụ của W10: **"đi cùng với"** (không "gây ra"), **"giữ các biến còn lại trong mô hình không đổi"** (biến chưa đo thì không được giữ), **đơn vị**, và **khoảng đã quan sát** của train (mô hình không có gì để nói ngoài khoảng đó). Khoảng tin cậy 95% dùng sai số chuẩn bền theo khối 14 ngày (kết luận của Phần C). Vì mục tiêu là log AQI, hệ số được đổi thành **% thay đổi của AQI ngày mai**.

In [ ]:
ct = coef_table(m_rob, cc, f3)
show = ct.assign(**{"β": ct["β (log AQI)"].round(3), "p (khối)": ct["p (khối)"].round(3)})[["biến", "β", "p (khối)", "diễn giải"]]
display(show.set_index("biến"))

### Đọc bảng

- **Ba biến thời tiết có khoảng tin cậy loại trừ 0 và dễ giải thích:** độ ẩm cao hơn đi cùng AQI ngày mai thấp hơn (−1,1% mỗi 10 điểm %), gió mạnh hơn đi cùng AQI thấp hơn (−1,8% mỗi m/s) và ngày có mưa đi cùng AQI thấp hơn nhiều nhất (−9,2% với 10 mm so với không mưa). Hướng khớp với kết quả của notebook 05 (độ ẩm nhất quán nhất; mưa rửa trôi).
- **Gió hướng Đông** đi cùng AQI **cao** hơn (+1,7% mỗi m/s), nhưng gió Bắc thì không tách khỏi 0. Hướng gió thay đổi theo thành phố, nên đây là gợi ý cần kiểm tra theo từng thành phố, chưa phải kết luận.
- **Nhiệt độ, lớp biên khí quyển, áp suất và mức đổi áp suất không tách khỏi 0** sau khi đã giữ AQI hôm nay và chất ô nhiễm. Lớp biên (+0,1% mỗi 100 m, từ −0,5% đến +0,7%) tiếp tục **không ủng hộ** trực giác "lớp biên thấp thì ô nhiễm bị nén lại", cùng kết luận với notebook 05.
- Các hệ số là **điều kiện** theo AQI hôm nay và chất ô nhiễm hôm nay. Thời tiết có thể đã tác động lên chúng, nên hệ số thời tiết ở đây nhỏ hơn tương quan thô ở notebook 05. Hai con số không mâu thuẫn, chúng trả lời hai câu hỏi khác nhau.
- Trong các chất ô nhiễm, PM2.5 có hệ số lớn nhất (+3,0% khi PM2.5 hôm nay cao hơn 10%), hợp lý vì PM2.5 thường là chất quyết định AQI ở các thành phố này. Hệ số của SO2 và NO2 không tách khỏi 0.
- Hệ số thành phố chỉ có nghĩa so với Delhi (mức tham chiếu).

# Phần E — Mở tập test (một lần duy nhất)

Từ đây mới dùng tập test (01/07/2019 – 29/02/2020). Mọi quyết định (đặc trưng, log, `alpha`, cấu hình cây) đã chốt ở các phần trên. Mô hình fit trên **toàn bộ train**, dự đoán test, đổi lại về điểm AQI bằng `exp`. Khoảng tin cậy của **hiệu số MAE** lấy bằng bootstrap theo khối 14 ngày lịch (ngày kề nhau, và các thành phố cùng ngày, đều không độc lập).

In [ ]:
pred_te, pred_lk, fitted = {}, {}, {}
for lv, f in LEVELS.items():
    m = make_pipe(f, "ridge", alpha=best_alpha[lv]).fit(tr[f + ["City"]], tr.y_log)      # fit trên train, một lần
    fitted[f"Ridge {lv}"] = m
    pred_te[f"Ridge {lv}"], pred_lk[f"Ridge {lv}"] = np.exp(m.predict(te[f + ["City"]])), np.exp(m.predict(lk[f + ["City"]]))
for lv in ["L1", "L2", "L3"]:
    f = LEVELS[lv]
    m = make_pipe(f, "hgb", **best_hgb[lv]).fit(tr[f + ["City"]], tr.y_log)
    pred_te[f"HGB {lv}"], pred_lk[f"HGB {lv}"] = np.exp(m.predict(te[f + ["City"]])), np.exp(m.predict(lk[f + ["City"]]))
pred_te.update(baseline_preds(tr, te)); pred_lk.update(baseline_preds(tr, lk))
PERSIST = "Ngày mai = hôm nay"
order = ["Trung bình theo thành phố", "Trung vị cùng tháng (khí hậu)", PERSIST, "Ridge L1", "Ridge L2", "Ridge L3", "Ridge L4", "HGB L1", "HGB L2", "HGB L3"]

def score_table(d, preds):
    heavy = d.y_poor.to_numpy()
    rows = {}
    for k in order:
        p = preds[k]
        rows[k] = {"MAE": mae(d.y, p), "RMSE": rmse(d.y, p), "R²": r2(d.y, p),
                   "MAE ngày nặng": mae(d.y[heavy], p[heavy]) if heavy.any() else np.nan, "MAE ngày còn lại": mae(d.y[~heavy], p[~heavy])}
    t = pd.DataFrame(rows).T
    t["giảm MAE so với 'hôm nay'"] = (1 - t.MAE / t.loc[PERSIST, "MAE"]) * 100
    return t
test_tab = score_table(te, pred_te)
display(test_tab.round(2).rename(columns={"giảm MAE so với 'hôm nay'": "giảm MAE so với 'hôm nay' (%)"}))

In [ ]:
bl = calendar_blocks(te.Date_target)
err = lambda k: np.abs(te.y.to_numpy() - pred_te[k])
pairs = [("Ridge L1", PERSIST), ("Ridge L2", "Ridge L1"), ("Ridge L3", "Ridge L2"), ("Ridge L3", "Ridge L1"), ("Ridge L3", PERSIST),
         ("Ridge L4", "Ridge L3"), ("HGB L3", "Ridge L3")]
rows = []
for a_, b_ in pairs:
    d_, lo, hi = boot_mae_diff(bl, err(a_), err(b_))
    rows.append({"so sánh (A trừ B)": f"{a_}  −  {b_}", "ΔMAE (điểm AQI)": d_, "KTC 95% thấp": lo, "KTC 95% cao": hi,
                 "kết luận": "A tốt hơn B" if hi < 0 else "A kém hơn B" if lo > 0 else "chưa tách khỏi 0"})
mdiff = {(a_, b_): (r["ΔMAE (điểm AQI)"], r["KTC 95% thấp"], r["KTC 95% cao"]) for (a_, b_), r in zip(pairs, rows)}
mae_diff = pd.DataFrame(rows).set_index("so sánh (A trừ B)")
display(mae_diff.round(2))

## Biểu đồ 2: thêm chất ô nhiễm rồi thời tiết giảm sai số trên tập test

In [ ]:
labels = {"Trung bình theo thành phố": "Luôn đoán trung bình của thành phố", "Trung vị cùng tháng (khí hậu)": "Trung vị cùng tháng (khí hậu)", PERSIST: "Ngày mai = hôm nay",
          "Ridge L1": "Mức 1: chỉ AQI các ngày trước", "Ridge L2": "Mức 2: + chất ô nhiễm hôm nay", "Ridge L3": "Mức 3: + thời tiết đến hôm nay",
          "Ridge L4": "Cận trên: + thời tiết ngày mai"}
keys = list(labels)
vals = [test_tab.loc[k, "MAE"] for k in keys]
fig, ax = plt.subplots(figsize=(10, 4.9))
cols = [NEUTRAL] * len(keys); cols[keys.index("Ridge L3")] = ACCENT
bars = ax.barh(range(len(keys)), vals, color=cols)
bars[-1].set_hatch("///"); bars[-1].set_facecolor("white"); bars[-1].set_edgecolor(NEUTRAL)
ax.bar_label(bars, labels=[vn(v, 1) for v in vals], padding=3, fontsize=9)
ax.axvline(test_tab.loc[PERSIST, "MAE"], color="#555555", lw=1, ls="--")
ax.set_yticks(range(len(keys))); ax.set_yticklabels([labels[k] for k in keys]); ax.invert_yaxis()
ax.set_xlabel("MAE trên tập test (điểm AQI; thấp hơn là tốt hơn)"); ax.set_xlim(0, max(vals) * 1.12)
m_p, m1, m2, m3, m4 = (test_tab.loc[k, "MAE"] for k in [PERSIST, "Ridge L1", "Ridge L2", "Ridge L3", "Ridge L4"])
assert m4 < m3 < m2 < m1 < m_p, "Thứ tự các mức đã đổi: sửa tiêu đề và phần đọc kết quả"
finish_chart(fig, ax, f"Trên tập test, thêm chất ô nhiễm rồi thời tiết hôm nay giảm sai số dự đoán AQI ngày mai từ {vn(m1, 1)} xuống {vn(m3, 1)} điểm; "
                      f"'ngày mai = hôm nay' sai {vn(m_p, 1)} điểm", te,
             note="Test = 01/07/2019 - 29/02/2020. Cột kẻ sọc: kịch bản dự báo thời tiết hoàn hảo, chỉ là cận trên.")
save_fig(fig, "02_mae_by_model")

### Đọc bảng test và bảng hiệu số

- **"Ngày mai = hôm nay" sai 23,5 điểm AQI** (R² 0,86). Mô hình mức 3 sai **19,5** (R² 0,91), tức **giảm 16,8%**, hiệu số −3,94 điểm (khoảng tin cậy −5,4 đến −2,6): tách khỏi 0 rõ ràng. RMSE cũng giảm từ 37,6 xuống 30,3.
- **Hai bậc thang tiếp theo đều tách khỏi 0:** mức 1 → mức 2 (thêm chất ô nhiễm) −2,0; mức 2 → mức 3 (thêm thời tiết) −1,4 (khoảng tin cậy −1,9 đến −0,9). **Riêng phần thời tiết giảm khoảng 1,4 điểm, tức khoảng 7% MAE của mức 2.** Khiêm tốn, đúng như dự đoán từ notebook 05 ("gió và lớp biên yếu hơn kỳ vọng").
- **Mô hình chỉ có lịch sử AQI (mức 1) chưa tách khỏi baseline** (−0,54; −1,2 đến +0,1). Phần lớn lợi thế so với "ngày mai = hôm nay" đến từ **chất ô nhiễm và thời tiết**, không phải từ việc xử lý lịch sử AQI khéo hơn.
- **Cận trên (biết trước thời tiết ngày mai) chỉ tốt hơn mức 3 thêm 0,43 điểm** (−0,62 đến −0,26). Nghĩa là nếu sau này có dự báo thời tiết thật, lợi ích thêm là nhỏ: thông tin quan trọng đã nằm ở trạng thái hôm nay.
- **Trên ngày nặng, sai số lớn hơn nhiều** (MAE 47,5 với "hôm nay" và 39,8 với mức 3, so với 17,5 và 14,5 ở ngày còn lại): mô hình giảm được sai số ở cả hai nhóm nhưng ngày nặng vẫn là chỗ khó nhất, đúng chỗ người dân cần cảnh báo.
- **Cây tăng cường mức 3 tốt hơn Ridge mức 3 trên test** (18,1 so với 19,5; hiệu số −1,4, khoảng tin cậy −2,1 đến −0,7), dù trên validation hai mô hình hoà. Có hai cách đọc: quan hệ phi tuyến có thật, hoặc test có đặc điểm riêng. **Đừng đổi mô hình chính sau khi đã thấy test**: làm vậy test không còn là test. Nên báo cả hai trong bảng, giữ Ridge làm mô hình chính vì giải thích được (W10), và nói rõ cây hơn ~1,4 điểm.

## Biểu đồ 3: sai số theo thành phố, so với 'ngày mai = hôm nay'

In [ ]:
byc = []
for c in CITIES:
    mk = (te.City == c).to_numpy()
    byc.append({"Thành phố": c, "n": int(mk.sum()), "ngày nặng": int(te.y_poor[mk].sum()),
                "MAE hôm nay": mae(te.y[mk], pred_te[PERSIST][mk]), "MAE mức 1": mae(te.y[mk], pred_te["Ridge L1"][mk]),
                "MAE mức 3": mae(te.y[mk], pred_te["Ridge L3"][mk])})
byc = pd.DataFrame(byc).sort_values("ngày nặng", ascending=False).set_index("Thành phố")
byc["giảm so với 'hôm nay' (%)"] = (1 - byc["MAE mức 3"] / byc["MAE hôm nay"]) * 100
display(byc.round(1))

fig, ax = plt.subplots(figsize=(9.5, 5))
yy, h = np.arange(len(byc)), 0.38
b1 = ax.barh(yy - h / 2, byc["MAE hôm nay"], h, color=NEUTRAL, label="Ngày mai = hôm nay")
b2 = ax.barh(yy + h / 2, byc["MAE mức 3"], h, color=ACCENT, label="Mức 3 (AQI, chất ô nhiễm, thời tiết)")
ax.bar_label(b1, labels=[vn(v, 1) for v in byc["MAE hôm nay"]], padding=2, fontsize=8); ax.bar_label(b2, labels=[vn(v, 1) for v in byc["MAE mức 3"]], padding=2, fontsize=8)
ax.set_yticks(yy); ax.set_yticklabels([f"{c} ({n} ngày nặng)" for c, n in zip(byc.index, byc["ngày nặng"])]); ax.invert_yaxis()
ax.set_xlabel("MAE trên tập test (điểm AQI)"); ax.legend(frameon=False, loc="lower right", fontsize=8.5)
dl, gu, lu = byc.loc["Delhi"], byc.loc["Gurugram"], byc.loc["Lucknow"]
south = byc.loc[["Bengaluru", "Chennai", "Hyderabad"]]
assert all(byc.loc[c, "MAE mức 3"] < byc.loc[c, "MAE hôm nay"] for c in ["Delhi", "Gurugram", "Lucknow"]), "Mức 3 không còn tốt hơn ở ba thành phố nặng: sửa tiêu đề"
finish_chart(fig, ax, f"Mô hình mức 3 giảm sai số ở cả ba thành phố nặng nhất: Delhi từ {vn(dl['MAE hôm nay'], 1)} xuống {vn(dl['MAE mức 3'], 1)} điểm, "
                      f"Lucknow từ {vn(lu['MAE hôm nay'], 1)} xuống {vn(lu['MAE mức 3'], 1)}, Gurugram từ {vn(gu['MAE hôm nay'], 1)} xuống {vn(gu['MAE mức 3'], 1)}", te,
             note="Thành phố xếp theo số ngày AQI >= 201 trong tập test.")
save_fig(fig, "03_mae_by_city")

### Đọc biểu đồ 3

- Mức giảm sai số cùng chiều ở **cả 8 thành phố** (7% đến 22%), lớn nhất ở ba thành phố nặng nhất (Delhi 20%, Lucknow 22%, Gurugram 19%).
- Sai số tuyệt đối ở miền Nam (Bengaluru, Hyderabad, Chennai) nhỏ hơn vì AQI ở đó thấp (xem `03_eda`); **đừng đọc nó như "mô hình tốt hơn ở miền Nam"**. Đơn vị so sánh công bằng giữa các thành phố là mức giảm so với baseline của chính thành phố đó.
- Delhi và Gurugram có thời tiết gần trùng nhau (xem `04_weather_merge`), nên hai kết quả gần giống nhau ở đây **không phải hai bằng chứng độc lập**. Phần I bỏ Gurugram để kiểm tra.

# Phần F — Cảnh báo ngày AQI ≥ 201 trước một ngày (W11)

## Chọn chỉ số và ngưỡng trước khi nhìn kết quả

- **Chỉ số chính: recall, ràng buộc bởi precision.** Bỏ sót một ngày nặng nghĩa là người dân không được cảnh báo sức khỏe; báo nhầm một ngày chỉ tốn một lần cảnh báo thừa. Giống sàng lọc bệnh (W11): recall quan trọng hơn, nhưng precision thấp quá thì cảnh báo mất uy tín. Đây là **một nhận định về chi phí**, không phải sự thật toán học; hình 4 cho phép người ra quyết định chọn điểm khác.
- **Ngưỡng:** áp ngưỡng lên **AQI dự đoán**, chọn bằng **F2** (recall nặng gấp đôi precision) trên các dòng validation của `TimeSeriesSplit` (dự đoán "ngoài mẫu" của train). Ngưỡng **không** chọn trên test (W11: chọn trên test là rò rỉ dưới dạng hấp dẫn nhất).
- **Công bằng với baseline:** "ngày mai = hôm nay" cũng được chọn ngưỡng theo đúng quy tắc và đúng các dòng đó. Bảng cũng báo ngưỡng 201 mặc định (baseline theo nghĩa đen).
- **Không dùng accuracy.** Mô hình "không bao giờ cảnh báo" đúng 80% số ngày trong test và có recall bằng 0 (hàng đầu của bảng).

In [ ]:
y_tr = tr.y_poor.to_numpy()
oof = {lv: oof_predict(tr, f, lambda f=f, lv=lv: make_pipe(f, "ridge", alpha=best_alpha[lv])) for lv, f in LEVELS.items()}
idx = oof["L1"][0]
assert all((oof[lv][0] == idx).all() for lv in LEVELS)               # mọi ngưỡng được chọn trên CÙNG các dòng validation
thr = {lv: best_threshold(y_tr[idx], oof[lv][1])[0] for lv in LEVELS}
thr_p = best_threshold(y_tr[idx], tr.AQI.to_numpy()[idx])[0]           # 'ngày mai = hôm nay' cũng được chọn ngưỡng công bằng như mô hình
oof_h = oof_predict(tr, LEVELS["L3"], lambda: make_pipe(LEVELS["L3"], "hgb", **best_hgb["L3"]))
assert (oof_h[0] == idx).all(); thr_h = best_threshold(y_tr[idx], oof_h[1])[0]
print(f"Validation: {len(idx)} dòng, {int(y_tr[idx].sum())} ngày nặng. Ngưỡng chọn bằng F2 (recall nặng gấp đôi precision):")
print({"Ngày mai = hôm nay": thr_p, **{f"Ridge {lv}": thr[lv] for lv in LEVELS}})

## Bảng cảnh báo trên tập test (ngưỡng 201 và ngưỡng chọn trên validation)

In [ ]:
y_te = te.y_poor.to_numpy()
aq_te = te.AQI.to_numpy()
flags = {"Không bao giờ cảnh báo": np.zeros(len(te), bool),
         f"{PERSIST} (ngưỡng 201)": aq_te >= 201, f"{PERSIST} (ngưỡng {thr_p:.0f})": aq_te >= thr_p}
for lv in LEVELS:
    flags[f"Ridge {lv} (ngưỡng 201)"] = pred_te[f"Ridge {lv}"] >= 201
    flags[f"Ridge {lv} (ngưỡng {thr[lv]:.0f})"] = pred_te[f"Ridge {lv}"] >= thr[lv]
flags["HGB L3 (ngưỡng 201)"], flags[f"HGB L3 (ngưỡng {thr_h:.0f})"] = pred_te["HGB L3"] >= 201, pred_te["HGB L3"] >= thr_h

def alert_row(flag, ci=True):
    m = alert_metrics(y_te, flag)
    r = {"recall": m["recall"], "precision": m["precision"], "F1": m["F1"], "số cảnh báo": m["n_canh_bao"], "bỏ sót": m["FN"], "báo nhầm": m["FP"]}
    if ci and m["TP"] > 0:
        _, lo, hi = boot_alert_ci(bl, y_te, flag, "recall"); r["recall KTC95%"] = f"[{lo:.2f}; {hi:.2f}]"
        _, lo, hi = boot_alert_ci(bl, y_te, flag, "precision"); r["precision KTC95%"] = f"[{lo:.2f}; {hi:.2f}]"
    return r
alert_tab = pd.DataFrame({k: alert_row(v) for k, v in flags.items()}).T
display(alert_tab.round(3))

## So sánh có khoảng tin cậy (bootstrap theo khối 14 ngày lịch)

In [ ]:
K = {"p201": f"{PERSIST} (ngưỡng 201)", "pt": f"{PERSIST} (ngưỡng {thr_p:.0f})",
     "L1t": f"Ridge L1 (ngưỡng {thr['L1']:.0f})", "L2t": f"Ridge L2 (ngưỡng {thr['L2']:.0f})", "Ht": f"HGB L3 (ngưỡng {thr_h:.0f})", "L3t": f"Ridge L3 (ngưỡng {thr['L3']:.0f})", "L3_201": "Ridge L3 (ngưỡng 201)", "L1_201": "Ridge L1 (ngưỡng 201)"}
cmp_pairs = [("L3t", "p201", "mô hình mức 3 (ngưỡng chọn) so với baseline đúng nghĩa đen"), ("L3t", "pt", "mức 3 so với 'hôm nay' cũng được chọn ngưỡng (so sánh công bằng)"),
             ("L3t", "L1t", "mức 3 so với mức 1: chất ô nhiễm và thời tiết thêm vào"), ("L3t", "L2t", "mức 3 so với mức 2: riêng phần thời tiết thêm vào"), ("L3_201", "p201", "cùng ngưỡng 201: mức 3 so với 'hôm nay'")]
rows = []
for a_, b_, what in cmp_pairs:
    for met in ["recall", "precision", "F1"]:
        d_, lo, hi = boot_alert_diff(bl, y_te, flags[K[a_]], flags[K[b_]], met)
        rows.append({"so sánh": what, "chỉ số": met, "Δ (A trừ B)": d_, "KTC 95% thấp": lo, "KTC 95% cao": hi,
                     "kết luận": "A hơn B" if lo > 0 else "A kém B" if hi < 0 else "chưa tách khỏi 0"})
alert_diff = pd.DataFrame(rows).set_index(["so sánh", "chỉ số"])
display(alert_diff.round(3))

### Đọc hai bảng

- **Ở cùng ngưỡng 201**, mức 3 hơn "ngày mai = hôm nay" ở cả ba chỉ số: recall 0,89 so với 0,86 (+0,033; 0,003 đến 0,068), precision 0,90 so với 0,85 (+0,047; 0,033 đến 0,065), F1 +0,040 (0,027 đến 0,055). Nghĩa là trong 389 ngày nặng, mô hình bỏ sót 43 ngày thay vì 56, và báo nhầm 40 ngày thay vì 59.
- **Khi cả hai được chọn ngưỡng cùng quy tắc** (so sánh công bằng), mức 3 bắt 96,9% ngày nặng ở ngưỡng 167, và "hôm nay" bắt 96,1% ở ngưỡng 160. **Recall không khác nhau** (+0,008; −0,010 đến +0,029). Cái hơn là **precision: +0,056 (0,035 đến 0,083)**, tức khoảng 5,6 điểm % báo đúng nhiều hơn; trong thực tế là 107 báo nhầm thay vì 143.
- **So với baseline "nghĩa đen" (ngưỡng 201), mô hình ở ngưỡng chọn bắt được nhiều hơn 11 điểm % ngày nặng nhưng báo đúng ít hơn 7 điểm %.** Đừng viết thành "mô hình hơn về mọi mặt": đó là hai điểm vận hành khác nhau trên cùng một đường cong. F1 chênh +0,011 (chưa tách khỏi 0).
- **Phần riêng của thời tiết không tách khỏi 0 ở bài toán cảnh báo**: mức 3 so với mức 2 cho recall 0,000, precision +0,003, F1 +0,002, cả ba khoảng tin cậy chứa 0. Ở MAE thì thời tiết giảm được 1,4 điểm; nhưng ở bước đặt ngưỡng 201 (chỉ cần biết trên hay dưới), lợi thế đó không còn thấy được trên một mùa đông. **Lợi thế cảnh báo so với "hôm nay" đến chủ yếu từ chất ô nhiễm (và lịch sử AQI), không từ thời tiết.** Đây là kết quả trung thực của đề tài, dù không phải kết quả mong muốn.
- Mức 4 (cận trên) và cây tăng cường có recall/precision gần mức 3 (HGB mức 3 ở ngưỡng 201: recall 0,91, precision 0,91), không đổi bức tranh.

## Biểu đồ 4: đường precision–recall trên tập test

In [ ]:
grid = np.arange(40, 401, 2)
def pr_curve(score):
    pts = [alert_metrics(y_te, score >= t) for t in grid]
    return np.array([[p["recall"], p["precision"]] for p in pts if p["n_canh_bao"] > 0])
curves = {PERSIST: (aq_te, thr_p, NEUTRAL, "--"), "Ridge L1": (pred_te["Ridge L1"], thr["L1"], "#7A8087", "-"), "Ridge L3": (pred_te["Ridge L3"], thr["L3"], ACCENT, "-")}
nm = {PERSIST: "Ngày mai = hôm nay", "Ridge L1": "Mức 1 (chỉ AQI)", "Ridge L3": "Mức 3 (+ chất ô nhiễm, thời tiết)"}
fig, ax = plt.subplots(figsize=(10, 5.6))
for k, (sc, t_star, col, ls) in curves.items():
    c = pr_curve(sc); ax.plot(c[:, 0], c[:, 1], color=col, ls=ls, lw=2, label=nm[k])
    for t_, filled in [(201, False), (t_star, True)]:
        mm = alert_metrics(y_te, sc >= t_)
        ax.scatter(mm["recall"], mm["precision"], s=55, facecolor=col if filled else "white", edgecolor=col, zorder=4, lw=1.6)
ax.axhline(y_te.mean(), color="#999999", lw=1, ls=":"); ax.text(0.99, y_te.mean() + 0.012, f"tỉ lệ ngày nặng = {vn(y_te.mean(), 2)} (precision của việc đoán bừa)", fontsize=8, color="#777777", ha="right")
ax.set_xlabel("Recall: trong các ngày nặng thật, tỉ lệ ngày được báo trước"); ax.set_ylabel("Precision: trong các ngày được báo, tỉ lệ đúng")
from matplotlib.lines import Line2D
ax.set_xlim(0.4, 1.01); ax.set_ylim(0.15, 1.02)
h, l = ax.get_legend_handles_labels()
h += [Line2D([], [], marker="o", ls="", mfc="white", mec="#555555", ms=7), Line2D([], [], marker="o", ls="", mfc="#555555", mec="#555555", ms=7)]
l += ["ngưỡng 201", "ngưỡng chọn trên validation (F2)"]
ax.legend(h, l, frameon=False, loc="lower left", fontsize=9)
r3, r_p = alert_metrics(y_te, flags[K["L3t"]]), alert_metrics(y_te, flags[K["pt"]])
assert r3["precision"] > r_p["precision"] and abs(r3["recall"] - r_p["recall"]) < 0.03, "Quan hệ recall/precision đã đổi: sửa tiêu đề"
finish_chart(fig, ax, f"Khi cả hai cách đều chọn ngưỡng theo cùng quy tắc, mô hình mức 3 bắt {vn(r3['recall'] * 100)}% ngày nặng với {vn(r3['precision'] * 100)}% báo đúng; "
                      f"'ngày mai = hôm nay' bắt {vn(r_p['recall'] * 100)}% với {vn(r_p['precision'] * 100)}% báo đúng", te,
             note="Test = 01/07/2019 - 29/02/2020; ngưỡng chọn trên các lần validation của train, không chọn trên test.")
save_fig(fig, "04_precision_recall")


### Đọc biểu đồ 4

- Đường của mức 3 (cam) **nằm trên** đường của "ngày mai = hôm nay" (xám nhạt) ở hầu hết khoảng recall từ 0,6 đến 0,95: ở cùng recall, báo đúng nhiều hơn. Hai đường chụm lại ở recall rất cao (trên 0,97), vì bắt gần hết ngày nặng thì mọi cách đều phải chấp nhận nhiều báo nhầm.
- Vòng tròn rỗng (ngưỡng 201) và chấm đặc (ngưỡng chọn bằng F2) cho thấy **cùng một mô hình có thể vận hành ở nhiều điểm**. Chọn điểm nào là quyết định của cơ quan môi trường (cảnh báo thừa tốn bao nhiêu so với bỏ sót), không phải của mô hình.
- Đường nằm xa đường ngang "đoán bừa" (precision 0,20): mọi cách đều học được điều gì đó thật, nhưng phần lớn đã có trong baseline.

## Theo nhóm thành phố

In [ ]:
groups = {"Delhi, Gurugram, Lucknow": ["Delhi", "Gurugram", "Lucknow"], "Jaipur, Amritsar": ["Jaipur", "Amritsar"], "Bengaluru, Chennai, Hyderabad": ["Bengaluru", "Chennai", "Hyderabad"]}
rows = []
for g_, cs in groups.items():
    mk = te.City.isin(cs).to_numpy()
    r = {"nhóm": g_, "n": int(mk.sum()), "ngày nặng": int(y_te[mk].sum())}
    for lab, fl in [("hôm nay@201", flags[K["p201"]]), ("hôm nay@ngưỡng", flags[K["pt"]]), ("mức 3@ngưỡng", flags[K["L3t"]])]:
        m = alert_metrics(y_te[mk], fl[mk])
        enough = r["ngày nặng"] >= 30                           # quy ước W12 của nhóm: dưới 30 ngày nặng thì không báo tỉ lệ
        r[f"recall {lab}"] = m["recall"] if enough else np.nan
        r[f"precision {lab}"] = m["precision"] if enough else np.nan
    rows.append(r)
group_tab = pd.DataFrame(rows).set_index("nhóm")
display(group_tab.round(2))
print("NaN = nhóm có dưới 30 ngày nặng trong tập test: không đủ để báo recall/precision.")

### Đọc bảng theo nhóm thành phố (W12: báo kết quả theo nhóm, không chỉ tổng thể)

- **Delhi, Gurugram, Lucknow (341 ngày nặng):** recall 0,99 và precision 0,82 ở ngưỡng chọn, tốt hơn tổng thể. Phần lớn điểm của bảng tổng nằm ở ba thành phố này.
- **Jaipur và Amritsar (chỉ 38 ngày nặng trong test):** recall 0,84 (**thấp hơn** "hôm nay" ở cùng ngưỡng là 0,89) và precision 0,52. Cảnh báo ở hai thành phố này kém tin cậy hơn nhiều, và mẫu nhỏ nên con số cũng chưa chắc. **Không nên nói "mô hình dùng được cho cả 8 thành phố".**
- **Bengaluru, Chennai, Hyderabad (10 ngày nặng):** không đủ để đánh giá cảnh báo (dưới 30 ngày nặng), nên không có số. Đây là một **giới hạn phải ghi vào model card**: mô hình cảnh báo chưa được kiểm chứng ở miền Nam, nơi hầu như không có ngày nặng.

# Phần G — Kiểm lại độ ẩm và nhiệt độ (phát hiện khám phá của notebook 05)

Notebook 05 chọn độ ẩm **sau khi nhìn kết quả**, nên cần kiểm lại. Cần nói rõ một điều: 05 phân tích toàn bộ khoảng chung 06/2017 – 07/2020, tức **đã gồm cả giai đoạn test**. Vì vậy giai đoạn test **không phải tập "chưa từng thấy" cho riêng phát hiện đó**. Cách làm trung thực hơn là chạy **đúng cách làm của 05** (lệch so với ngày điển hình cùng thành phố, cùng tháng; tertile trong từng tháng) trên **hai giai đoạn tách rời**, train và test, rồi xem hai giai đoạn có cùng chiều không. Phần này cố ý dùng **bảng đã làm sạch hồi cứu** (`load_with_weather()`, đúng như notebook 05), không dùng bảng dự báo dựng từ raw: đây là phép lặp lại một phát hiện mô tả của 05, không tạo đặc trưng hay nhãn cho mô hình. Kiểm chứng thật sự "ngoài mẫu" là ở Phần D và E: hệ số độ ẩm ở Phần D chỉ fit trên train (âm, loại trừ 0), và Phần E cho thấy thêm thời tiết giảm sai số trên test.

In [ ]:
from corr_utils import add_anomalies, tertile_in_strata
eda_w, win_w, S0, E0 = load_with_weather(verbose=False)
dw = win_w.dropna(subset=["AQI"]).copy()
core, n_heavy_all = core_cities(dw)                                   # 5 thành phố đủ ngày nặng (quy ước C1 của notebook 03)
periods = {"train (06/2017-06/2019)": dw[dw.Date <= TRAIN_END], "test (07/2019-02/2020)": dw[(dw.Date >= TEST_START) & (dw.Date <= TEST_END)]}

def replicate(d):
    # đúng cách làm của notebook 05 trên MỘT giai đoạn: lệch so với ngày điển hình cùng thành phố, cùng tháng; tertile độ ẩm trong từng (thành phố, tháng)
    a = add_anomalies(d, ["AQI", "humidity_mean", "temp_mean"])
    rows = []
    for c in core:
        g = a[a.City == c].copy()
        g["blk"] = block_id(g).to_numpy(); g["t"] = tertile_in_strata(g, "humidity_mean").to_numpy()
        rh = boot_corr(g.AQI_a, g.humidity_mean_a, g.blk); rt = boot_corr(g.AQI_a, g.temp_mean_a, g.blk)
        lo_, hi_, pz = (g.t == "thấp").to_numpy(), (g.t == "cao").to_numpy(), g.poor.to_numpy()
        d_, dlo, dhi = block_boot(g.blk.to_numpy(), np.column_stack([pz & lo_, lo_, pz & hi_, hi_]), lambda s: (s[:, 0] / s[:, 1] - s[:, 2] / s[:, 3]) * 100)
        rows.append({"Thành phố": c, "n ngày": len(g), "ngày nặng": int(pz.sum()),
                     "ρ độ ẩm": rh["rho"], "ρ độ ẩm KTC": f"[{rh['lo']:.2f}; {rh['hi']:.2f}]", "ρ nhiệt độ": rt["rho"], "ρ nhiệt độ KTC": f"[{rt['lo']:.2f}; {rt['hi']:.2f}]",
                     "% nặng, ẩm thấp": g[lo_].poor.mean() * 100, "% nặng, ẩm cao": g[hi_].poor.mean() * 100, "chênh (điểm %)": d_, "chênh KTC": f"[{dlo:.0f}; {dhi:.0f}]"})
    return pd.DataFrame(rows).set_index("Thành phố")
recheck = {k: replicate(v) for k, v in periods.items()}
for k, v in recheck.items():
    print(k); display(v.round(2))

## Biểu đồ 5: tỉ lệ ngày AQI ≥ 201 theo nhóm độ ẩm, hai giai đoạn

In [ ]:
three = [c for c in core if all(recheck[k].loc[c, "ngày nặng"] >= 30 for k in recheck)]       # đủ 30 ngày nặng ở CẢ hai giai đoạn
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8), sharey=True)
for ax, (k, d_) in zip(axes, periods.items()):
    a_ = add_anomalies(d_, ["humidity_mean"]); xs = np.arange(len(three)); w_ = 0.26
    for q, (g_, cc_, nm_) in enumerate(zip(["thấp", "giữa", "cao"], [ACCENT, "#7A8087", NEUTRAL], ["Ẩm thấp (1/3 khô nhất cùng tháng)", "Ẩm vừa", "Ẩm cao (1/3 ẩm nhất)"])):
        vals_ = []
        for c in three:
            g = a_[a_.City == c]; vals_.append(g[tertile_in_strata(g, "humidity_mean") == g_].poor.mean() * 100)
        b = ax.bar(xs + (q - 1) * w_, vals_, w_, color=cc_, label=nm_); ax.bar_label(b, labels=[vn(v) for v in vals_], padding=2, fontsize=8)
    ax.set_xticks(xs); ax.set_xticklabels(three); ax.set_ylim(0, 100); ax.set_xlabel(k)
axes[0].set_ylabel("% ngày có AQI >= 201"); axes[0].legend(frameon=False, ncol=1, fontsize=8, loc="upper right")
tr_k, te_k = list(recheck)
d_tr, d_te = recheck[tr_k].loc["Delhi"], recheck[te_k].loc["Delhi"]
assert all(recheck[k].loc[c, "chênh (điểm %)"] > 0 for k in recheck for c in three), "Chiều đã đổi: sửa tiêu đề"
finish_chart(fig, axes[0], f"Cùng tháng, ngày ẩm thấp có nhiều ngày AQI >= 201 hơn ngày ẩm cao ở cả hai giai đoạn: Delhi {vn(d_tr['% nặng, ẩm thấp'])}% so với {vn(d_tr['% nặng, ẩm cao'])}% (train), "
                         f"{vn(d_te['% nặng, ẩm thấp'])}% so với {vn(d_te['% nặng, ẩm cao'])}% (test), cùng chiều ở Gurugram và Lucknow", pd.concat(periods.values()),
             note="Tertile độ ẩm tính trong từng (thành phố, tháng) của chính giai đoạn đó, đúng cách làm của notebook 05. Chỉ vẽ thành phố có từ 30 ngày nặng ở cả hai giai đoạn.")
save_fig(fig, "05_humidity_recheck")


### Đọc bảng và biểu đồ 5

- **Độ ẩm: chiều âm lặp lại ở cả hai giai đoạn và cả 5 thành phố** (ρ từ −0,10 đến −0,39 ở train, từ −0,15 đến −0,32 ở test). Khoảng tin cậy loại trừ 0 ở 8/10 trường hợp; hai trường hợp còn lại là Lucknow (ρ −0,10 và −0,15, khoảng tin cậy chứa 0).
- **Nhiệt độ: chiều dương lặp lại ở 9/10 trường hợp** (ngoại lệ Lucknow ở test, ρ −0,01). Như ở 05, chỉ nên nói về chiều, không nói về độ lớn.
- **Tỉ lệ ngày nặng ở nhóm ẩm thấp cao hơn nhóm ẩm cao ở Delhi, Gurugram và Lucknow, ở cả hai giai đoạn** (Delhi +18 điểm % ở cả train lẫn test). Nhưng **khoảng tin cậy của chênh lệch chứa 0 ở 5/6 trường hợp** (chỉ Delhi ở train tách khỏi 0): **chiều nhất quán, độ lớn chưa chắc**. Con số "64% so với 38%" ở notebook 05 là tổng hợp cả khoảng chung và không nên trích như ước lượng độ lớn đáng tin; nên nói "cao hơn một chút đến khá nhiều tuỳ thành phố".
- Giống 05: độ ẩm cao có thể đại diện cho mưa, sương mù, mây hoặc một khối khí khác. Kết quả vẫn là **"đi cùng"**, không phải "gây ra".

# Phần H — Giai đoạn phong toả 03–07/2020 (tách riêng, không dùng để chọn gì)

Mô hình fit trên train (trước phong toả) rồi dự đoán 975 cặp (thành phố, ngày) từ 01/03 đến 01/07/2020. Mục đích: xem mô hình có lệch hệ thống khi nguồn phát thải đổi đột ngột không. Chỉ 18 ngày nặng nên **không tính recall và precision**.

In [ ]:
heavy_lk = lk.y_poor.to_numpy()
lock_tab = pd.DataFrame({k: {"MAE": mae(lk.y, pred_lk[k]), "RMSE": rmse(lk.y, pred_lk[k]), "AQI dự đoán TB": pred_lk[k].mean()}
                         for k in [PERSIST, "Ridge L1", "Ridge L2", "Ridge L3", "Ridge L4"]}).T
lock_tab.loc["Thực tế", ["AQI dự đoán TB"]] = lk.y.mean()
display(lock_tab.round(1))
print(f"{len(lk)} cặp (thành phố, ngày) từ {lk.Date_target.min():%d/%m/%Y} đến {lk.Date_target.max():%d/%m/%Y}; chỉ {int(heavy_lk.sum())} ngày AQI >= 201 ({heavy_lk.mean()*100:.1f}%) nên KHÔNG tính recall/precision.")
print(f"Tập test cùng chỉ số (để so): MAE hôm nay {test_tab.loc[PERSIST,'MAE']:.1f}, mức 3 {test_tab.loc['Ridge L3','MAE']:.1f}; AQI thực tế TB {te.y.mean():.1f}")

### Đọc bảng

- Sai số **thấp hơn hẳn** giai đoạn test (MAE 13,8 so với 19,5 của mức 3) vì AQI trong phong toả thấp và ổn định (trung bình ~97 so với 139 ở test). Đừng đọc là "mô hình tốt hơn".
- Mức 3 vẫn hơn "hôm nay" (13,8 so với 14,8), cùng chiều với test nhưng chênh nhỏ.
- **Không có dấu hiệu lệch hệ thống**: AQI dự đoán trung bình 98,7 so với thực tế 96,6. Mô hình không "hoảng" vì phát thải giảm đột ngột, một phần vì lịch sử AQI hôm nay đã cập nhật mức nền mới.

# Phần I — Kiểm tra độ nhạy (cho câu vấn đáp "nếu đổi cách làm, kết luận có đổi không?")

Năm kiểm tra, đều đo **hiệu số MAE** vì đó là phát hiện chính (mức 3 so với mức 1 và so với "hôm nay"):
1. **Bỏ ngẫu nhiên 10% dòng train**, lặp 100 lần (câu vấn đáp W8).
2. **Bỏ Gurugram** (thời tiết gần trùng Delhi, không phải bằng chứng độc lập).
3. **So đường dự báo từ raw với đường hồi cứu cũ** (kiểm tra quyết định thiết kế 3 và việc đổi cohort): cùng một giao thức, ba biến thể A, B, C.
4. **Tách test thành hai giai đoạn**: 07–11/2019 (mưa và sau mưa) và 12/2019–02/2020 (mùa đông).
5. **Bỏ hai biến mùa** (sin, cos): thời tiết còn giúp không nếu không có mùa đi kèm.

In [ ]:
fit_mae = lambda d_tr, d_te, lv: mae(d_te.y, np.exp(make_pipe(LEVELS[lv], "ridge", alpha=best_alpha[lv]).fit(d_tr[LEVELS[lv] + ["City"]], d_tr.y_log).predict(d_te[LEVELS[lv] + ["City"]])))
rng = np.random.default_rng(42)
drops = []
for _ in range(100):
    s = tr.sample(frac=0.9, random_state=int(rng.integers(1e9))).sort_index()
    a1, a3 = fit_mae(s, te, "L1"), fit_mae(s, te, "L3")
    drops.append((a3 - a1, a3 - mae(te.y, te.AQI)))
drops = np.array(drops)
print("Bỏ ngẫu nhiên 10% dòng train, 100 lần (câu vấn đáp W8):")
display(pd.DataFrame({"ΔMAE mức 3 trừ mức 1": [drops[:, 0].min(), np.median(drops[:, 0]), drops[:, 0].max()],
                      "ΔMAE mức 3 trừ 'hôm nay'": [drops[:, 1].min(), np.median(drops[:, 1]), drops[:, 1].max()]}, index=["nhỏ nhất", "trung vị", "lớn nhất"]).round(2))

In [ ]:
ng = ds[ds.City != "Gurugram"]
trg, teg = ng[ng.split == "train"], ng[ng.split == "test"]
blg = calendar_blocks(teg.Date_target)
e_p, e1, e3 = (np.abs(teg.y.to_numpy() - teg.AQI.to_numpy()),
               *[np.abs(teg.y.to_numpy() - np.exp(make_pipe(LEVELS[lv], "ridge", alpha=best_alpha[lv]).fit(trg[LEVELS[lv] + ["City"]], trg.y_log).predict(teg[LEVELS[lv] + ["City"]]))) for lv in ["L1", "L3"]])
r1, r2_ = boot_mae_diff(blg, e3, e_p), boot_mae_diff(blg, e3, e1)
print(f"Bỏ Gurugram (thời tiết gần trùng Delhi): MAE hôm nay {e_p.mean():.2f}, mức 1 {e1.mean():.2f}, mức 3 {e3.mean():.2f}")
print(f"  mức 3 − hôm nay: {r1[0]:+.2f} (KTC 95% {r1[1]:+.2f} đến {r1[2]:+.2f}) · mức 3 − mức 1: {r2_[0]:+.2f} ({r2_[1]:+.2f} đến {r2_[2]:+.2f})")

In [ ]:
# Đường dự báo từ raw (dùng trong notebook này) so với đường hồi cứu cũ (bảng đã làm sạch trên toàn chuỗi), CÙNG một giao thức:
# mỗi biến thể tự chọn alpha (TimeSeriesSplit trong train) và ngưỡng F2 (dự đoán ngoài mẫu của train); không lựa chọn nào nhìn test.
#   A = mã cũ, cohort của nó · B = mã mới trên cùng cohort với A (bỏ các mẫu chỉ đường mới có, theo diff thực tế) · C = mã mới, cohort raw
cmp = fu.compare_paths(ds, ds_old, alphas=ALPHAS)
display(cmp["table"].T)
print("Hiệu MAE so cặp giữa các biến thể (chỉ tính khi CÙNG khoá và CÙNG nhãn), bootstrap theo khối 14 ngày lịch:")
display(cmp["paired"].round(4))
fdiff, fcells = fu.feature_diff(ds, ds_old)
n_common = int(fdiff["khoá chung"].iloc[0])
print(f"Trên {n_common:,} khoá chung của hai đường: AQI hôm nay khác {int(fdiff.loc['AQI', 'ô khác'])} ô, nhãn khác {int(fdiff.loc['y', 'ô khác'])} ô. "
      f"Các cột đặc trưng có ô khác nhau (dung sai 1e-9):")
display(fdiff.loc[(fdiff["ô khác"] > 0) & ~fdiff.index.isin(["AQI", "y"]),
                  ["ô khác", "khác do một bên NaN", "max |lệch| (hai bên có giá trị)", "ô khác: train", "ô khác: test", "ô khác: lockdown"]])
run_a, run_b, run_c = (cmp["runs"][k] for k in cmp["table"].index)
# Biến thể C phải chính là kết quả chính của notebook (cùng cohort, cùng alpha, cùng ngưỡng, cùng MAE): so sánh này không dùng một mô hình khác
assert run_c["n"] == {"train": len(tr), "test": len(te), "lockdown": len(lk)} and run_c["alpha"] == {lv: best_alpha[lv] for lv in ["L1", "L2", "L3"]}
assert all(abs(run_c["mae"]["test"][lv] - test_tab.loc[f"Ridge {lv}", "MAE"]) < 1e-9 for lv in ["L1", "L2", "L3"]), "biến thể C phải trùng kết quả chính của notebook"
assert run_c["thr"] == {PERSIST: thr_p, **{lv: thr[lv] for lv in ["L1", "L2", "L3"]}}
# B dùng đúng cohort của A; C nhiều hơn A đúng bằng số khoá thêm/bớt đo được ở Phần A (không giả định trước con số)
assert run_b["n"] == run_a["n"] and run_c["n"]["train"] + run_c["n"]["test"] + run_c["n"]["lockdown"] == sum(run_a["n"].values()) + n_new - n_lost

In [ ]:
mo = te.Date_target.dt.month
per = {"07-11/2019 (mưa, sau mưa)": (mo >= 7) & (mo <= 11), "12/2019-02/2020 (mùa đông)": (mo == 12) | (mo <= 2)}
rows = []
for name, mk in per.items():
    mk = mk.to_numpy()
    rows.append({"giai đoạn test": name, "n": int(mk.sum()), "ngày nặng": int(y_te[mk].sum()), "MAE hôm nay": mae(te.y[mk], pred_te[PERSIST][mk]),
                 "MAE mức 1": mae(te.y[mk], pred_te["Ridge L1"][mk]), "MAE mức 3": mae(te.y[mk], pred_te["Ridge L3"][mk])})
subp = pd.DataFrame(rows).set_index("giai đoạn test"); display(subp.round(2))

In [ ]:
no_cal = lambda lv: [x for x in LEVELS[lv] if x not in CAL_FEATS]
def fit_nc(lv):
    f = no_cal(lv)
    return mae(te.y, np.exp(make_pipe(f, "ridge", alpha=best_alpha[lv]).fit(tr[f + ["City"]], tr.y_log).predict(te[f + ["City"]])))
nc1, nc3 = fit_nc("L1"), fit_nc("L3")
print(f"Bỏ hai biến mùa (sin, cos): MAE mức 1 {nc1:.2f}, mức 3 {nc3:.2f}, chênh {nc3 - nc1:+.2f}  (có biến mùa: {test_tab.loc['Ridge L1','MAE']:.2f}, {test_tab.loc['Ridge L3','MAE']:.2f}, chênh {test_tab.loc['Ridge L3','MAE'] - test_tab.loc['Ridge L1','MAE']:+.2f})")

### Đọc kết quả độ nhạy

- **Bỏ 10% dòng train:** hiệu số mức 3 − mức 1 chỉ dao động từ −3,48 đến −3,30 qua 100 lần, và mức 3 − "hôm nay" từ −4,00 đến −3,83. Kết luận không phụ thuộc vào vài dòng cụ thể. (Biên độ nhỏ vì đây là xáo trộn của train, không phải của test; độ không chắc của test nằm ở khoảng tin cậy bootstrap theo khối ở Phần E.)
- **Bỏ Gurugram:** mức 3 vẫn hơn "hôm nay" 3,34 điểm (−4,61 đến −2,10) và hơn mức 1 là 2,81 điểm (−3,90 đến −1,77). Kết quả **không do việc đếm Delhi và Gurugram hai lần**.
- **Đường dự báo từ raw so với đường hồi cứu cũ** (A = mã cũ; B = mã mới trên đúng cohort của A; C = mã mới trên cohort raw, tức kết quả chính của notebook). Trên test, mức 3: đổi cách dựng đặc trưng trên cùng cohort (B − A) làm MAE đổi +0,0002 (−0,0069 đến +0,0071); thêm 5 mẫu train (C − B) đổi +0,0093 (−0,0033 đến +0,0213); cộng lại (C − A) +0,0095 (−0,0052 đến +0,0237). Mọi khoảng tin cậy, ở test và ở 03–07/2020, cả ba mức, đều chứa 0. Ba biến thể có cùng khoá và cùng nhãn ở hai tập đó nên đây là so cặp trên cùng một outcome. `alpha` không đổi (10; 0,1; 30). Ngưỡng F2 của mức 3 đổi từ 166 (A, B) lên 167 (C): số báo nhầm ở ngưỡng chọn giảm từ 113 xuống 107, số ngày nặng bắt được vẫn là 377. Trên 8.663 khoá chung, AQI hôm nay và nhãn trùng nhau; đặc trưng chỉ khác ở train (63 ô: 35 ô CO, 22 ô trung bình 7 ngày, 6 ô AQI hôm qua và hôm kia), còn ở test và 03–07/2020 trùng từng ô. Nghĩa là trên snapshot này **việc sửa rò rỉ gần như không đổi sai số đo trên test**; chỗ đổi nhìn thấy được là ngưỡng cảnh báo, do train thêm 5 mẫu. Đây là một kết quả đo, không phải lý do để giữ đường cũ: đường cũ vẫn sai về nguyên tắc vì đặc trưng của train phụ thuộc vào các ngày sau.
- **Hai giai đoạn của test:** lợi thế của mức 3 có ở cả hai, nhưng **lớn hơn ở mùa đông** (MAE 28,2 → 22,1, giảm 21%) so với mưa và sau mưa (20,7 → 18,0, giảm 13%). Nghĩa là phần lớn giá trị nằm ở mùa đông, đúng lúc khó và cần cảnh báo nhất.
- **Bỏ biến mùa:** hiệu số mức 3 − mức 1 gần như không đổi (−3,51 so với −3,40). Thời tiết **không chỉ đóng vai trò thay mùa**.

# Tổng hợp

## Trả lời câu hỏi nghiên cứu (phần dự đoán và cảnh báo)

**Có, nhưng khiêm tốn, và phần đóng góp riêng của thời tiết nhỏ hơn phần của chất ô nhiễm.**

- Dự đoán AQI ngày mai: mô hình có thời tiết giảm sai số **16,8%** so với "ngày mai = hôm nay" (23,5 → 19,5 điểm). Riêng thời tiết giảm thêm khoảng **1,4 điểm** (~7%) sau khi đã có AQI và chất ô nhiễm.
- Cảnh báo ngày AQI ≥ 201: ở cùng ngưỡng 201, mô hình hơn baseline ở cả recall, precision và F1. Ở ngưỡng chọn công bằng, mô hình **báo đúng hơn ~5,6 điểm %** với cùng recall. **Phần riêng của thời tiết không tách khỏi 0** ở bài toán cảnh báo.

## Phát hiện (mỗi ý gắn một biểu đồ hoặc bảng)

| # | Phát hiện (có con số) | Hình / bảng | Confounder có thể | Hạn chế |
|---|---|---|---|---|
| P1 | "Ngày mai = hôm nay" sai 23,5 điểm AQI trên test (R² 0,86), so với 64,5 (trung bình thành phố) và 43,4 (khí hậu cùng tháng). Mô hình chỉ có lịch sử AQI (22,9) chưa tách khỏi baseline (−0,5; −1,2 đến +0,1). | 2 | Quán tính của chính chuỗi AQI | Baseline mạnh nên mọi chênh lệch đều nhỏ về tuyệt đối |
| P2 | Hai bậc thang đều tách khỏi 0: + chất ô nhiễm −2,0; + thời tiết −1,4 (−1,9 đến −0,9); tổng so với "hôm nay" −3,94 (−5,4 đến −2,6), giảm 16,8%. Biết trước thời tiết ngày mai chỉ thêm −0,4. | 2, bảng E | Mùa (đã có biến mùa; bỏ biến mùa không đổi kết luận) | Một mùa đông test; thời tiết là tái phân tích ERA5, không phải số đo tại trạm |
| P3 | Mức giảm sai số cùng chiều ở cả 8 thành phố, lớn nhất ở Delhi (36,7 → 29,3), Lucknow (29,1 → 22,7), Gurugram (37,1 → 30,1). Trên ngày nặng: 47,5 → 39,8. | 3 | Delhi và Gurugram không độc lập (đã kiểm tra bằng cách bỏ Gurugram) | Sai số tuyệt đối miền Nam nhỏ vì AQI thấp, không so trực tiếp giữa thành phố |
| P4 | Hệ số thời tiết (train, sai số chuẩn bền): độ ẩm −1,1% mỗi 10 điểm %, gió −1,8% mỗi m/s, mưa 10 mm −9,2%. Nhiệt độ, lớp biên, áp suất không tách khỏi 0. | Bảng D | Hệ số có điều kiện theo AQI và chất ô nhiễm hôm nay; VIF nhiệt độ 8,7 | "Đi cùng với", không phải nguyên nhân; hệ số AQI trễ không diễn giải riêng lẻ (VIF đến 14,5) |
| P5 | Cảnh báo ở ngưỡng 201: mức 3 recall 0,89, precision 0,90 so với 0,86 và 0,85 của "hôm nay" (F1 +0,040; 0,027 đến 0,055). Ngưỡng chọn công bằng: precision +0,056 (0,035 đến 0,083), recall chưa khác (+0,008). Mức 3 so với mức 2: cả ba chỉ số chứa 0. | 4, bảng F | Mức nền ô nhiễm giảm theo thời gian (25% → 20% ngày nặng) | Ngưỡng chọn bằng F2 là một nhận định chi phí; chỉ một mùa đông |
| P6 | Theo nhóm: Delhi, Gurugram, Lucknow (341 ngày nặng) recall 0,99, precision 0,82. Jaipur và Amritsar (38 ngày nặng) recall 0,84, precision 0,52, **kém hơn**. Miền Nam (10 ngày nặng): không đủ để đánh giá. | Bảng F (nhóm) | Nguồn phát thải khác nhau theo vùng (bụi, đốt rơm rạ) | Không nên nói mô hình dùng được cho cả 8 thành phố |
| P7 | Độ ẩm: ρ âm ở cả 5 thành phố và cả hai giai đoạn (8/10 loại trừ 0); nhiệt độ ρ dương 9/10. Chênh tỉ lệ ngày nặng (ẩm thấp − ẩm cao) dương ở Delhi, Gurugram, Lucknow ở cả hai giai đoạn nhưng khoảng tin cậy chứa 0 ở 5/6. | 5, bảng G | Mưa, sương mù, mây (độ ẩm là đại diện) | 05 đã dùng cả giai đoạn test nên đây không phải kiểm chứng độc lập; chiều chắc, độ lớn chưa |
| P8 | Phong toả: MAE 14,8 ("hôm nay") và 13,8 (mức 3); AQI dự đoán trung bình 98,7 so với thực 96,6: không lệch hệ thống. | Bảng H | Phát thải đổi đột ngột | Chỉ 18 ngày nặng: không đánh giá cảnh báo |
| P9 | Cây tăng cường mức 3 tốt hơn Ridge mức 3 trên test (18,1 so với 19,5; −1,4; −2,1 đến −0,7) dù hoà trên validation. | Bảng E | Phi tuyến; đặc điểm riêng của test | Không đổi mô hình chính sau khi thấy test; báo cả hai |

## Hạn chế (cụ thể, gắn với kết luận nó đe doạ)

- **Chỉ một mùa đông trong test (07/2019 – 02/2020).** Khoảng tin cậy bootstrap theo khối cho biết sai số lấy mẫu trong mùa đó, nhưng **không** cho biết mô hình sẽ ra sao ở mùa đông 2020–21 (khác về thời tiết và phát thải). Đe doạ P2, P5.
- **Train chỉ có hai mùa đông** (2017–18 và 2018–19), nên cả việc học lẫn việc chọn ngưỡng dựa trên hai mùa. Đe doạ P5.
- **Tỉ lệ ngày nặng giảm theo thời gian** (25,0% train → 20,0% test → 1,8% phong toả). Ngưỡng chọn trên train có thể hơi lệch cho giai đoạn sau. Đe doạ P5.
- **Thời tiết là tái phân tích ERA5 trên ô lưới 0,25°, không phải số đo tại trạm quan trắc.** Dùng thời tiết thực tế đến hôm nay không phải là dự báo; khi triển khai thật phải thay bằng dự báo của ngày mai, mà kịch bản "cận trên" cho thấy lợi ích thêm vào chỉ khoảng 0,4 điểm. Đe doạ P2.
- **Nhãn là AQI như được ghi trong raw của snapshot hiện có**, gồm cả những điểm mà bước làm sạch hồi cứu coi là nghi lỗi. Không có bản phát hành theo ngày của CPCB nên không mô phỏng được việc số liệu bị sửa về sau, và raw không được coi là số đo đã kiểm định tại trạm. Đe doạ mọi con số sai số.
- **Cặp ngày thiếu AQI trong raw bị loại** (236 trên 8.904 cặp của khoảng chung: 174 thiếu AQI hôm nay, 62 thiếu AQI ngày mai), nên không dự đoán và không đánh giá được cho những ngày đó. Nếu thiếu AQI tập trung vào ngày nặng (cảm biến quá tải) thì mô hình trông tốt hơn thực tế. Chưa kiểm tra được.
- **Miền Nam không có ngày nặng**, nên cảnh báo chưa được kiểm chứng ở đó; **Jaipur và Amritsar** chỉ có 38 ngày nặng. Đe doạ P6.
- **Mô hình hụt ở các đợt tăng đột ngột** (đuôi phải nặng ở QQ-plot). Với bài toán cảnh báo sức khỏe, đây là đúng loại sai sót đắt nhất.
- **Chỉ số AQI là chỉ số tổng hợp theo ngày của CPCB** (theo thang Ấn Độ), nên kết quả không chuyển nguyên sang thang AQI của nước khác.

## Biến thể để các bạn trong nhóm thử (tránh bốn phiên bản khác nhau: giữ nguyên `model_utils.py`, chỉ thử trong notebook riêng)

1. Thêm biến trễ của thời tiết (hôm qua, hai ngày trước) hoặc trung bình trượt 3 ngày, và xem phần riêng của thời tiết ở bài toán cảnh báo có tách khỏi 0 không.
2. Mô hình riêng cho nhóm Delhi–Gurugram–Lucknow (hỗn hợp một mô hình cho 8 thành phố có thể đang làm Jaipur và Amritsar kém đi).
3. Đổi tiêu chí chọn ngưỡng (F1, hoặc recall cố định 0,95 rồi tối đa precision) và xem hình 4 đổi ra sao.
4. Giữ Ridge nhưng thêm vài tương tác thành phố × độ ẩm hoặc thành phố × gió, rồi so với cây tăng cường.

## Kiểm tra lại các con số trong phần viết, rồi lưu tóm tắt lần chạy

In [ ]:
# Các con số mà phần viết nhắc tới. Chúng là kết quả THỰC NGHIỆM của một snapshot đầu vào (sha256 ở ô Setup) và giao thức RA-DEC-02,
# đo lại trên đường dự báo từ raw. Đúng snapshot thì phải khớp (lệch là lỗi, phát ra ở ô cuối); đầu vào khác snapshot thì chỉ đối chiếu
# và ghi lại chỗ lệch. Bất biến (thứ tự thời gian, rò rỉ, cohort đếm lại từ raw) đã assert ở Phần A và không đi qua đây.
# Dung sai: các con số tác giả đã kiểm từ trước giữ nguyên dung sai cũ; con số mới thêm dùng nửa đơn vị của chữ số cuối trong phần viết.
# Mỗi nhóm con số khai báo nó phụ thuộc đầu vào nào (`needs`): chỉ khi các đầu vào ĐÓ đúng snapshot thì mới bắt buộc khớp.
RAW, WX, CLEAN = "raw", "thời tiết theo ngày", "bảng sạch hồi cứu (chỉ để đối chiếu)"
FC, RETRO = [RAW, WX], [CLEAN, WX]                               # đường dự báo: raw + thời tiết · Phần G và đường hồi cứu: bảng sạch + thời tiết
T, PAIR = test_tab, lambda a_, b_: mdiff[(a_, b_)]
items = [(f"MAE test, {k}", T.loc[k, "MAE"], v, 0.05) for k, v in [(PERSIST, 23.47), ("Ridge L1", 22.93), ("Ridge L2", 20.92), ("Ridge L3", 19.53), ("Ridge L4", 19.10),
                                                                 ("Trung bình theo thành phố", 64.54), ("Trung vị cùng tháng (khí hậu)", 43.44), ("HGB L3", 18.09)]]
items += [("R² test, 'hôm nay'", T.loc[PERSIST, "R²"], 0.86, 0.006), ("R² test, mức 3", T.loc["Ridge L3", "R²"], 0.91, 0.006),
          ("RMSE test, 'hôm nay'", T.loc[PERSIST, "RMSE"], 37.61, 0.1), ("RMSE test, mức 3", T.loc["Ridge L3", "RMSE"], 30.25, 0.1),
          ("giảm MAE của mức 3 so với 'hôm nay' (%)", T.loc["Ridge L3", "giảm MAE so với 'hôm nay'"], 16.8, 0.2),
          ("MAE ngày nặng, 'hôm nay'", T.loc[PERSIST, "MAE ngày nặng"], 47.46, 0.1), ("MAE ngày nặng, mức 3", T.loc["Ridge L3", "MAE ngày nặng"], 39.77, 0.1),
          ("MAE ngày còn lại, 'hôm nay'", T.loc[PERSIST, "MAE ngày còn lại"], 17.48, 0.1), ("MAE ngày còn lại, mức 3", T.loc["Ridge L3", "MAE ngày còn lại"], 14.48, 0.1)]
for (a_, b_), want in {("Ridge L1", PERSIST): (-0.54, -1.20, 0.07), ("Ridge L2", "Ridge L1"): (-2.01, -3.04, -1.05), ("Ridge L3", "Ridge L2"): (-1.39, -1.92, -0.92),
                       ("Ridge L3", PERSIST): (-3.94, -5.39, -2.56), ("Ridge L4", "Ridge L3"): (-0.43, -0.62, -0.26), ("HGB L3", "Ridge L3"): (-1.44, -2.09, -0.73)}.items():
    items += [(f"ΔMAE test {a_} − {b_}: {lab}", x, v, 0.06) for lab, x, v in zip(["hiệu", "KTC thấp", "KTC cao"], PAIR(a_, b_), want)]

B = byc
for c, want in {"Delhi": (36.7, 29.3), "Lucknow": (29.1, 22.7), "Gurugram": (37.1, 30.1)}.items():
    items += [(f"MAE test {c}, 'hôm nay'", round(float(B.loc[c, "MAE hôm nay"]), 1), want[0], None), (f"MAE test {c}, mức 3", round(float(B.loc[c, "MAE mức 3"]), 1), want[1], None)]
items += [("mức giảm nhỏ nhất theo thành phố (%)", B["giảm so với 'hôm nay' (%)"].min(), 15.0, 8.0), ("mức giảm lớn nhất theo thành phố (%)", B["giảm so với 'hôm nay' (%)"].max(), 15.0, 8.0),
          ("MAE validation, Ridge mức 3", round(float(cv_tab.loc[f"Ridge L3: {LEVEL_NAME['L3']}", "MAE validation"]), 1), 22.3, None),
          ("MAE validation, 'hôm nay'", float(np.mean(cv_base[PERSIST])), 25.96, 0.05)]

ad = lambda i, m: alert_diff.loc[(cmp_pairs[i][2], m), "Δ (A trừ B)"]
items += [("cùng ngưỡng 201, mức 3 − 'hôm nay': recall", ad(4, "recall"), 0.033, 0.005), ("cùng ngưỡng 201: precision", ad(4, "precision"), 0.047, 0.005), ("cùng ngưỡng 201: F1", ad(4, "F1"), 0.040, 0.005),
          ("ngưỡng chọn công bằng, mức 3 − 'hôm nay': recall", ad(1, "recall"), 0.008, 0.005), ("ngưỡng chọn công bằng: precision", ad(1, "precision"), 0.056, 0.005),
          ("số cảnh báo của 'hôm nay' ở ngưỡng 201", int(flags[K["p201"]].sum()), 392, None), ("số cảnh báo của mức 3 ở ngưỡng 201", int(flags["Ridge L3 (ngưỡng 201)"].sum()), 386, None),
          ("ngưỡng F2 của 'hôm nay'", thr_p, 160.0, None), ("ngưỡng F2 của mức 3", thr["L3"], 167.0, None),
          ("báo nhầm của mức 3 ở ngưỡng chọn", int(alert_tab.loc[K["L3t"], "báo nhầm"]), 107, None), ("báo nhầm của 'hôm nay' ở ngưỡng chọn", int(alert_tab.loc[K["pt"], "báo nhầm"]), 143, None)]

g1 = group_tab.loc["Delhi, Gurugram, Lucknow"]; g2 = group_tab.loc["Jaipur, Amritsar"]
items += [("Delhi-Gurugram-Lucknow, mức 3 ở ngưỡng chọn: recall", g1["recall mức 3@ngưỡng"], 0.99, 0.01), ("Delhi-Gurugram-Lucknow: precision", g1["precision mức 3@ngưỡng"], 0.82, 0.01),
          ("Jaipur-Amritsar, mức 3 ở ngưỡng chọn: recall", g2["recall mức 3@ngưỡng"], 0.84, 0.01), ("Jaipur-Amritsar: precision", g2["precision mức 3@ngưỡng"], 0.52, 0.01),
          ("Jaipur-Amritsar, 'hôm nay' ở ngưỡng chọn: recall", g2["recall hôm nay@ngưỡng"], 0.89, 0.01),
          ("ngày nặng ở miền Nam trong test", int(group_tab.loc["Bengaluru, Chennai, Hyderabad", "ngày nặng"]), 10, None)]

rk = list(recheck)
items_g = [("Phần G: số trường hợp ρ nhiệt độ dương (trên 10)", int(sum((recheck[k]["ρ nhiệt độ"] > 0).sum() for k in rk)), 9, None),
           ("Phần G: Delhi, chênh tỉ lệ ngày nặng ẩm thấp − ẩm cao, train (điểm %)", recheck[rk[0]].loc["Delhi", "chênh (điểm %)"], 18.3, 0.2),
           ("Phần G: Delhi, chênh tỉ lệ ngày nặng ẩm thấp − ẩm cao, test (điểm %)", recheck[rk[1]].loc["Delhi", "chênh (điểm %)"], 18.6, 0.2)]
items += [("03-07/2020: MAE 'hôm nay'", lock_tab.loc[PERSIST, "MAE"], 14.8, 0.1), ("03-07/2020: MAE mức 3", lock_tab.loc["Ridge L3", "MAE"], 13.8, 0.1),
          ("03-07/2020: AQI dự đoán trung bình của mức 3", lock_tab.loc["Ridge L3", "AQI dự đoán TB"], 98.7, 0.2), ("03-07/2020: AQI thực tế trung bình", lock_tab.loc["Thực tế", "AQI dự đoán TB"], 96.6, 0.2),
          ("bỏ Gurugram: ΔMAE mức 3 − 'hôm nay'", r1[0], -3.34, 0.06), ("bỏ Gurugram: ΔMAE mức 3 − mức 1", r2_[0], -2.81, 0.06),
          ("bỏ biến mùa: ΔMAE mức 3 − mức 1", nc3 - nc1, -3.51, 0.06)]
chk.expect_values(items, needs=FC)
chk.expect_values(items_g, needs=RETRO)

# Con số mà phần viết nhắc tới nhưng trước đây chưa được đối chiếu (Phần A, B, C, D, F, I), và các con số của phép so hai đường
h2, h3 = 0.006, 0.0006                                           # nửa đơn vị của chữ số thập phân thứ 2 và thứ 3 (cộng một chút sai số số học)
cvm = lambda k: float(cv_tab.loc[k, "MAE validation"])
beta = ct.set_index("biến")["β (log AQI)"]
eff = lambda f, k: float((np.exp(beta[f] * k) - 1) * 100)        # % thay đổi của AQI ngày mai khi biến tăng k đơn vị
a1 = acf1.copy(); a1.index = a1.index.astype(str)
ret_all = ret.loc["Cộng"]
chk.expect_values([("cặp (thành phố, ngày t) trong khoảng chung", int(ret_all.iloc[0]), 8904, None), ("cặp thiếu AQI(t)", int(ret_all["thiếu AQI(t)"]), 174, None),
                   ("cặp có AQI(t), thiếu AQI(t+1)", int(ret_all["có AQI(t), thiếu AQI(t+1)"]), 62, None), ("số mẫu giữ lại", len(ds), 8668, None),
                   ("số ngày cắt trong phép thử rò rỉ", len(leak), 23, None), ("tỉ lệ ngày nặng của train (%)", float(tr.y_poor.mean() * 100), 25.0, 0.06)], needs=[RAW])
chk.expect_values([("số mẫu của đường hồi cứu", len(ds_old), 8663, None)], needs=RETRO)
new_items = [("MAE validation, trung bình theo thành phố", float(np.mean(cv_base["Trung bình theo thành phố"])), 60.54, h2),
             ("MAE validation, khí hậu cùng tháng", float(np.mean(cv_base["Trung vị cùng tháng (khí hậu)"])), 56.12, h2),
             ("MAE validation, Ridge mức 1", cvm(f"Ridge L1: {LEVEL_NAME['L1']}"), 25.52, h2), ("MAE validation, Ridge mức 2", cvm(f"Ridge L2: {LEVEL_NAME['L2']}"), 23.72, h2),
             ("MAE validation, cây mức 2", cvm("Cây tăng cường (HGB) L2"), 22.94, h2), ("MAE validation, cây mức 3", cvm("Cây tăng cường (HGB) L3"), 22.43, h2),
             ("độ lệch giữa các lần chia, Ridge mức 1", float(cv_tab.loc[f"Ridge L1: {LEVEL_NAME['L1']}", "± giữa các lần chia"]), 1.38, h2),
             ("Spearman(|phần dư|, giá trị khớp), mục tiêu AQI", float(funnel(m_raw)), 0.23, h2), ("Spearman(|phần dư|, giá trị khớp), mục tiêu log AQI", float(funnel(m_log)), -0.03, h2),
             ("Breusch-Pagan LM, mục tiêu AQI", float(bp_raw[0]), 420.1, 0.6), ("Breusch-Pagan LM, mục tiêu log AQI", float(bp_log[0]), 316.05, 0.6),
             ("Durbin-Watson, log AQI", float(durbin_watson(m_log.resid)), 1.84, h2), ("R² hiệu chỉnh trong mẫu", float(m_log.rsquared_adj), 0.89, h2),
             ("AQI khớp nhỏ nhất của mô hình trên AQI", float(m_raw.fittedvalues.min()), -150.0, 20.0), ("phần dư log lớn nhất", float(m_log.resid.max()), 1.2, 0.06),
             ("ACF bậc 1 nhỏ nhất, bảy thành phố ngoài Hyderabad", float(a1.drop("Hyderabad").min()), 0.01, h2), ("ACF bậc 1 lớn nhất, bảy thành phố ngoài Hyderabad", float(a1.drop("Hyderabad").max()), 0.19, h2),
             ("ACF bậc 1, Hyderabad", float(a1["Hyderabad"]), 0.34, h2),
             ("VIF lớn nhất", float(vif.max()), 14.5, 0.06), ("VIF nhỏ nhất trong bốn biến AQI trễ", float(vif[["aqi_0", "aqi_1", "aqi_2", "aqi_wk"]].min()), 9.8, 0.06),
             ("VIF nhiệt độ", float(vif["temp_mean"]), 8.7, 0.06), ("VIF PM2.5", float(vif["pm25_0"]), 8.3, 0.06),
             ("Cook's distance lớn nhất", float(cook.max()), 0.077, h3), ("số dòng có Cook's distance vượt 4/n", int((cook > 4 / len(cook)).sum()), 345, None),
             ("hệ số độ ẩm (% mỗi 10 điểm %)", eff("humidity_mean", 10), -1.1, 0.06), ("hệ số tốc độ gió (% mỗi m/s)", eff("wind_speed_mean", 1), -1.8, 0.06),
             ("hệ số mưa (% với 10 mm)", eff("precip_log", np.log1p(10)), -9.2, 0.06), ("hệ số gió Đông (% mỗi m/s)", eff("wind_u", 1), 1.7, 0.06),
             ("hệ số lớp biên (% mỗi 100 m)", eff("blh_mean", 100), 0.1, 0.06), ("hệ số PM2.5 (% khi cao hơn 10%)", float((1.1 ** beta["pm25_0"] - 1) * 100), 3.0, 0.06),
             ("mức 3 ở ngưỡng chọn − 'hôm nay' ở ngưỡng 201: recall", ad(0, "recall"), 0.113, h3), ("... precision", ad(0, "precision"), -0.071, h3), ("... F1", ad(0, "F1"), 0.011, h3),
             ("mức 3 − mức 2 ở ngưỡng chọn: recall", ad(3, "recall"), 0.000, h3), ("mức 3 − mức 2: precision", ad(3, "precision"), 0.003, h3), ("mức 3 − mức 2: F1", ad(3, "F1"), 0.002, h3),
             ("recall của mức 3 ở ngưỡng chọn", float(alert_tab.loc[K["L3t"], "recall"]), 0.969, h3), ("recall của 'hôm nay' ở ngưỡng chọn", float(alert_tab.loc[K["pt"], "recall"]), 0.961, h3),
             ("bỏ sót của mức 3 ở ngưỡng 201", int(alert_tab.loc[K["L3_201"], "bỏ sót"]), 43, None), ("báo nhầm của mức 3 ở ngưỡng 201", int(alert_tab.loc[K["L3_201"], "báo nhầm"]), 40, None),
             ("bỏ sót của 'hôm nay' ở ngưỡng 201", int(alert_tab.loc[K["p201"], "bỏ sót"]), 56, None), ("báo nhầm của 'hôm nay' ở ngưỡng 201", int(alert_tab.loc[K["p201"], "báo nhầm"]), 59, None),
             ("bỏ 10% train: ΔMAE mức 3 − mức 1, nhỏ nhất", float(drops[:, 0].min()), -3.48, h2), ("bỏ 10% train: ΔMAE mức 3 − mức 1, lớn nhất", float(drops[:, 0].max()), -3.30, h2),
             ("bỏ 10% train: ΔMAE mức 3 − 'hôm nay', nhỏ nhất", float(drops[:, 1].min()), -4.00, h2), ("bỏ 10% train: ΔMAE mức 3 − 'hôm nay', lớn nhất", float(drops[:, 1].max()), -3.83, h2),
             ("bỏ Gurugram: KTC của mức 3 − 'hôm nay', thấp", r1[1], -4.61, h2), ("bỏ Gurugram: KTC của mức 3 − 'hôm nay', cao", r1[2], -2.10, h2),
             ("bỏ Gurugram: KTC của mức 3 − mức 1, thấp", r2_[1], -3.90, h2), ("bỏ Gurugram: KTC của mức 3 − mức 1, cao", r2_[2], -1.77, h2),
             ("test mùa đông: MAE 'hôm nay'", float(subp.iloc[1]["MAE hôm nay"]), 28.17, h2), ("test mùa đông: MAE mức 3", float(subp.iloc[1]["MAE mức 3"]), 22.14, h2),
             ("test mưa và sau mưa: MAE 'hôm nay'", float(subp.iloc[0]["MAE hôm nay"]), 20.67, h2), ("test mưa và sau mưa: MAE mức 3", float(subp.iloc[0]["MAE mức 3"]), 17.98, h2),
             ("có biến mùa: ΔMAE mức 3 − mức 1", float(T.loc["Ridge L3", "MAE"] - T.loc["Ridge L1", "MAE"]), -3.40, h2)]
chk.expect_values(new_items, needs=FC)

# Phép so hai đường (Phần I, kiểm tra 3): A = đường hồi cứu cũ, B = mã mới trên cohort của A, C = mã mới trên cohort raw
pz = cmp["paired"]
pzr = lambda who, part, lv: pz[(pz["so sánh (A trừ B)"] == who) & (pz["tập"] == part) & (pz["mức"] == lv)].iloc[0]
cmp_items = []
for who, want in {"B. − A.": (0.0002, -0.0069, 0.0071), "C. − B.": (0.0093, -0.0033, 0.0213), "C. − A.": (0.0095, -0.0052, 0.0237)}.items():
    r_ = pzr(who, "test", "L3")
    cmp_items += [(f"so đường {who} mức 3, test: {lab}", float(r_[col]), v, 0.00006) for lab, col, v in zip(["hiệu MAE", "KTC thấp", "KTC cao"], ["ΔMAE", "KTC 95% thấp", "KTC 95% cao"], want)]
tA, tC = cmp["table"].iloc[0], cmp["table"].iloc[2]
cmp_items += [("đường cũ (A): ngưỡng F2 của mức 3", float(tA["ngưỡng F2 L3"]), 166.0, None), ("đường cũ (A): mức 3 ở ngưỡng chọn, TP/FP/FN", tA["L3@F2 TP/FP/FN"], "377/113/12", None),
              ("đường mới (C): mức 3 ở ngưỡng chọn, TP/FP/FN", tC["L3@F2 TP/FP/FN"], "377/107/12", None),
              ("số khoá chung của hai đường", n_common, 8663, None),
              ("ô đặc trưng khác giữa hai đường: CO", int(fdiff.loc["co_0", "ô khác"]), 35, None), ("... trung bình 7 ngày", int(fdiff.loc["aqi_wk", "ô khác"]), 22, None),
              ("... AQI hôm qua và hôm kia", int(fdiff.loc[["aqi_1", "aqi_2"], "ô khác"].sum()), 6, None),
              ("tổng số ô đặc trưng khác giữa hai đường", int(fdiff.drop(index=["AQI", "y"])["ô khác"].sum()), 63, None)]
chk.expect_values(cmp_items)                                     # phụ thuộc cả ba đầu vào

# Kết luận định tính mà phần viết dựa vào (cũng là kết quả thực nghiệm của snapshot, không phải bất biến)
E = chk.expect
E((B["MAE mức 3"] < B["MAE hôm nay"]).all(), "mức 3 tốt hơn 'hôm nay' ở cả 8 thành phố", needs=FC)
E(best_alpha == {"L1": 10, "L2": 0.1, "L3": 30, "L4": 10}, f"alpha chọn trên validation = {best_alpha}, snapshot L1 10, L2 0,1, L3 30, L4 10", needs=FC)
E(alert_diff.loc[(cmp_pairs[1][2], "recall"), "kết luận"] == "chưa tách khỏi 0" and alert_diff.loc[(cmp_pairs[1][2], "precision"), "kết luận"] == "A hơn B",
  "ngưỡng chọn công bằng: recall chưa tách khỏi 0, precision của mức 3 hơn", needs=FC)
E(alert_diff.loc[(cmp_pairs[0][2], "F1"), "kết luận"] == "chưa tách khỏi 0", "mức 3 ở ngưỡng chọn so với baseline nghĩa đen: F1 chưa tách khỏi 0", needs=FC)
E((alert_diff.loc[cmp_pairs[3][2]].iloc[:, -1] == "chưa tách khỏi 0").all(), "mức 3 so với mức 2 ở bài toán cảnh báo: cả ba chỉ số chứa 0", needs=FC)
E(group_tab.loc["Bengaluru, Chennai, Hyderabad"].iloc[2:].isna().all(), "miền Nam: dưới 30 ngày nặng nên không báo recall/precision", needs=FC)
E(r1[2] < 0 and r2_[2] < 0, "bỏ Gurugram: hai khoảng tin cậy của ΔMAE vẫn nằm dưới 0", needs=FC)
E(subp["MAE mức 3"].lt(subp["MAE hôm nay"]).all(), "mức 3 tốt hơn 'hôm nay' ở cả hai giai đoạn của test", needs=FC)
E(all((recheck[k]["ρ độ ẩm"] < 0).all() for k in rk), "Phần G: ρ độ ẩm âm ở cả 5 thành phố và cả hai giai đoạn", needs=RETRO)
E(bool(pz["cùng khoá"].all() and pz["cùng nhãn"].all()), "ba biến thể A/B/C có cùng khoá và cùng nhãn ở test và ở 03-07/2020 (so cặp cùng outcome)")
E(int(fdiff.loc["AQI", "ô khác"]) == 0 and int(fdiff.loc["y", "ô khác"]) == 0, "trên khoá chung của hai đường, AQI hôm nay và nhãn trùng nhau ở mọi tập")
E(bool(((pz["KTC 95% thấp"] < 0) & (pz["KTC 95% cao"] > 0)).all()), "mọi hiệu MAE giữa A, B, C (test và 03-07/2020, ba mức) có khoảng tin cậy chứa 0")
E(int(fdiff["ô khác: test"].sum()) == 0 and int(fdiff["ô khác: lockdown"].sum()) == 0, "đặc trưng của test và 03-07/2020 trùng giữa hai đường; khác biệt chỉ nằm ở train")
E(all(cmp["table"][f"alpha {lv}"].nunique() == 1 for lv in ["L1", "L2", "L3"]), "alpha của ba biến thể A, B, C giống nhau ở cả ba mức")
chk.summary();

In [ ]:
# Lưu tóm tắt lần chạy cho người kiểm (ghi ra outputs/nb06/, không ghi vào data/ hay reports/). Có dự đoán theo từng khoá
# (thành phố, ngày) của test và 03-07/2020 để tính lại mọi chỉ số, kể cả so cặp, mà không cần chạy lại notebook.
OUT = data_paths.nb06_out_dir()
def pred_table(d, preds):
    t = d[["City", "Date", "Date_target", "split", "y", "y_poor"]].copy()
    for k in order:
        t[k] = preds[k]
    return t
preds_out = pd.concat([pred_table(te, pred_te), pred_table(lk, pred_lk)], ignore_index=True)
alerts_out = pd.DataFrame({k: {**{m: alert_metrics(y_te, v)[m] for m in ["TP", "FP", "FN", "TN", "recall", "precision", "F1"]}} for k, v in flags.items()}).T.rename_axis("quy tắc cảnh báo (test)")
summary = {
    "protocol": fu.PROTOCOL, "environment": ENV,
    "inputs": {k: {"path": data_paths.show(v), "sha256": chk.actual[k], "trùng snapshot": chk.matches[k]} for k, v in INPUTS.items()},
    "snapshot_checks": {"đầu vào đúng snapshot": chk.on_snapshot, "số con số đã đối chiếu": chk.n_checked, "số ở chế độ bắt buộc khớp": chk.n_strict,
                        "lệch ở chế độ bắt buộc khớp": chk.failures, "lệch được ghi lại (đầu vào khác snapshot)": chk.deviations},
    "cohort": {"n": {"train": len(tr), "test": len(te), "lockdown": len(lk)},
               "ngày nặng": {"train": int(tr.y_poor.sum()), "test": int(te.y_poor.sum()), "lockdown": int(lk.y_poor.sum())},
               "bảng giữ mẫu": ret.to_dict("index"), "chỉ có ở đường dự báo": n_new, "chỉ có ở đường hồi cứu": n_lost},
    "rò rỉ từ raw": {"số ngày cắt": len(leak), "ô khác khi cắt raw ở t": int(leak["ô khác (cắt raw ở t)"].sum()), "ô khác khi nhiễu dữ liệu sau t": int(leak["ô khác (nhiễu dữ liệu sau t)"].sum())},
    "alpha": best_alpha, "hgb": best_hgb, "validation": {"n": int(len(idx)), "ngày nặng": int(y_tr[idx].sum())},
    "ngưỡng F2": {PERSIST: thr_p, **{f"Ridge {lv}": thr[lv] for lv in LEVELS}, "HGB L3": thr_h},
    "MAE validation": cv_tab["MAE validation"].to_dict(),
    "test": test_tab.to_dict("index"),
    "hiệu MAE test": {f"{a_} − {b_}": {"hiệu": v[0], "KTC thấp": v[1], "KTC cao": v[2]} for (a_, b_), v in mdiff.items()},
    "cảnh báo test": alerts_out.to_dict("index"),
    "hiệu cảnh báo test": alert_diff.reset_index().to_dict("records"),
    "03-07/2020": lock_tab.to_dict("index"),
    "so đường (A/B/C)": cmp["table"].to_dict("index"),
    "so đường, hiệu MAE so cặp": cmp["paired"].to_dict("records"),
}
data_paths.save_json(summary, OUT / "nb06_run_summary.json", "tóm tắt lần chạy NB06")
data_paths.save_table(preds_out, OUT / "nb06_predictions.csv", "dự đoán theo khoá (test và 03-07/2020)", index=False)
data_paths.save_table(alerts_out, OUT / "nb06_alerts_test.csv", "bảng cảnh báo trên test (TP/FP/FN/TN)")
data_paths.save_table(ret, OUT / "nb06_retention.csv", "bảng giữ mẫu")
data_paths.save_table(cdiff, OUT / "nb06_cohort_diff.csv", "khoá thêm/bớt so với đường hồi cứu", index=False)
data_paths.save_table(cmp["table"].T, OUT / "nb06_path_comparison.csv", "so đường A/B/C")
data_paths.save_table(cmp["paired"], OUT / "nb06_path_comparison_paired.csv", "so đường A/B/C: hiệu MAE so cặp (test và 03-07/2020)", index=False)
data_paths.save_table(fcells, OUT / "nb06_feature_diff_cells.csv", "ô đặc trưng khác giữa hai đường (khoá chung)", index=False)
data_paths.save_table(chk.table(), OUT / "nb06_snapshot_checks.csv", "mọi con số đã đối chiếu (đo được / giá trị snapshot)", index=False)

# Lỗi (nếu có) phát ra Ở ĐÂY, sau khi các bảng trên đã được lưu: đầu vào đúng snapshot mà có con số lệch thì notebook dừng với AssertionError.
chk.finish()
print("Mọi con số mà phần viết nhắc tới khớp với kết quả vừa tính." if not chk.deviations else
      "Có đầu vào KHÁC snapshot: có con số lệch khỏi phần viết (danh sách ở trên và trong nb06_snapshot_checks.csv).")

## Việc còn lại cho báo cáo cuối kỳ (W12)

Notebook này đã có sẵn các phần cần cho checklist: kết quả **theo nhóm thành phố** (Phần E và F), **hạn chế** ở trên, và các nhóm có thể bị đại diện thiếu (miền Nam, Jaipur, Amritsar). Chưa làm ở đây: **model card một trang**, **mục "khai báo AI" trong README** và ethics checklist đầy đủ; các số liệu cho model card có thể lấy từ bảng ở Phần E, F và mục Hạn chế.